# WRAP_P6_CORRELATION_LAB — Notebook 06

**Spec ref:** §9.4 — Train + validate the correlation engine.

Answers: *how well can L1+L2 features predict the L3 pattern label the omniscient teacher saw?*

## Section 00 — Env & Imports

In [1]:
%load_ext autoreload
%autoreload 2
import os, glob, sys, subprocess, pathlib

# ── Load p6lab profile from the shell script ─────────────────────────────
PROFILE = "nq-on"  # ← edit me; matches a name in p6lab/scripts/p6lab_env.sh

_repo  = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "p6lab" / "scripts" / "p6lab_env.sh").is_file():
    _repo = _repo.parent
_script = _repo / "p6lab" / "scripts" / "p6lab_env.sh"
assert _script.is_file(), f"p6lab_env.sh not found searching up from {pathlib.Path.cwd()}"

_env = subprocess.run(
    ["bash", "-c", f"source {_script} {PROFILE} >/dev/null 2>&1 && env"],
    capture_output=True, text=True, cwd=_repo,
)
for line in _env.stdout.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        if k.startswith(("P6LAB_", "OMP_", "MKL_", "OPENBLAS_", "NUMEXPR_")):
            os.environ[k] = v

# ── Verify ──────────────────────────────────────────────────────────────
print("python:", sys.executable)
print("cwd:", os.getcwd())
print("PROFILE:", PROFILE)
print("P6LAB_DATA_FILE:", os.environ.get("P6LAB_DATA_FILE"))
print("P6LAB_SYMBOL:", os.environ.get("P6LAB_SYMBOL"))
print("OMP_NUM_THREADS:", os.environ.get("OMP_NUM_THREADS"))

# Comma-separated multi-file + glob support
specs = [s.strip() for s in os.environ.get("P6LAB_DATA_FILE", "").split(",") if s.strip()]
files = sorted({m for spec in specs for m in glob.glob(spec)})
print(f"{len(files)} files matched")
for f in files[:5]:
    print(" ", f)

import p6lab, databento
print("p6lab:", p6lab.__file__)
print("databento:", databento.__version__)
print("P6LAB_MAX_SNAPSHOTS:", os.environ.get("P6LAB_MAX_SNAPSHOTS"))


python: <path>
cwd: <path>
PROFILE: nq-on
P6LAB_DATA_FILE: <path>
P6LAB_SYMBOL: NQ
OMP_NUM_THREADS: None
1 files matched
  <path>
p6lab: <path>
databento: ▒.▒▒.0
P6LAB_MAX_SNAPSHOTS: None


In [2]:
import sys, logging, json, pickle
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

# ── shared config ─────────────────────────────────────────────────────────────
_NB_DIR = Path('<path>')
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))
import _common  # bootstraps sys.path; exposes NOTEBOOK_DATA_SLICE, helpers
import importlib, _common; importlib.reload(_common)

from _common import (
    NOTEBOOK_DATA_SLICE, HORIZONS, PURGE_ROWS, TIER_CUTOFFS,
    BASELINE_MIN_AUC_IMPROVEMENT, collect_snapshots, versioned_dir,
)
_DS       = NOTEBOOK_DATA_SLICE          # single source of truth
SYMBOL    = _DS['symbol']
TICK_SIZE = _DS['tick_size']

_P6LAB        = Path(_common._P6LAB_ROOT)
ARTIFACTS_DIR = _P6LAB / 'artifacts' / 'p6lab'
MAX_SNAPSHOTS = _DS['max_snapshots'] if _DS['max_snapshots'] is not None else float('inf')
logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)
np.random.seed(42)

print(f'Data  : {_DS["data_file"]}')
print(f'Symbol: {SYMBOL}  tick={TICK_SIZE}  max_snapshots={MAX_SNAPSHOTS}')
import pickle
MODELS_DIR    = _P6LAB / 'correlation_runs' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
BOOK_SHAPE_DIM = 40

import os
from threadpoolctl import threadpool_info
print("data_file:", _DS['data_file'])
print("exists?  :", os.path.isfile(_DS['data_file']) if isinstance(_DS['data_file'], str) else None)
print("env P6LAB_DATA_FILE:", os.environ.get("P6LAB_DATA_FILE"))
print("env P6LAB_DATA_DIR :", os.environ.get("P6LAB_DATA_DIR"))
print("Env vars (should all be unset):")
for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    print(f"  {v}: {os.environ.get(v, '(unset)')}")

print("\nActive thread pools (should show high num_threads):")
for p in threadpool_info():
    print(f"  {p['user_api']:8} {p['prefix']:20} num_threads={p['num_threads']}")


from p6lab.features._l1_adapter import L1Adapter, L1AdapterConfig
from p6lab.features.l1_features import L1FeatureNames, compute_l1_features
from p6lab.features.fragility_index import FragilityIndex
from p6lab.features.l2_features import (
    L2FeatureNames, L2History, L2Snapshot, compute_l2_features, compute_book_shape_vector,
)
from p6lab.correlation.engine import CorrelationEngine
from p6lab.patterns.template_matcher import TemplateMatcher
from p6lab.validation.cpcv import CascadeAwareCPCV
from p6lab.validation.information_gain import must_beat_baseline
import lightgbm as lgb
import time, psutil, os
_t0 = time.time()
_p = psutil.Process(os.getpid())
print(f"START rss={_p.memory_info().rss/1e9:.1f}GB")





Data  : <path>
Symbol: NQ  tick=▒.▒▒  max_snapshots=500000
data_file: <path>
exists?  : True
env P6LAB_DATA_FILE: <path>
env P6LAB_DATA_DIR : None
Env vars (should all be unset):
  OMP_NUM_THREADS: (unset)
  MKL_NUM_THREADS: (unset)
  OPENBLAS_NUM_THREADS: (unset)
  NUMEXPR_NUM_THREADS: (unset)

Active thread pools (should show high num_threads):
  blas     libscipy_openblas    num_threads=8
START rss=▒.▒▒GB


## Section 01 — Load Triple-View + Recompute Features

Reload snapshots (triple-view parquet stores events but we need the L1/L2 feature vectors to train). Recomputes L1 (16-dim) + L2 (12-dim) + FI (2-dim composites) per snapshot.

In [3]:
from functools import cache
import hashlib, json
from pathlib import Path
# Bump SCHEMA_VERSION whenever feature COMPUTATION changes WITHOUT changing
# column names (e.g. the FI column-index fix, level_persistence constant→varying).
# Column adds/renames are caught automatically by the feature-name fingerprint
# in _cache_path_for. Either way the cache key changes, so newer code can never
# silently load a stale parquet — the fast (cache) path is then guaranteed to
# equal what the slow (cold-build) path would produce.
SCHEMA_VERSION = 2  # bumped: FI now resolves L2 cols by name (DF/CF fix)

def _cache_path_for(_ds):
    # Cache key = dataset-slice params AND a fingerprint of the feature schema
    # (column set + SCHEMA_VERSION). A feature change → new key → cold rebuild,
    # instead of reloading an outdated cache.
    from p6lab.features.l1_features import L1FeatureNames as _L1
    from p6lab.features.l2_features import L2FeatureNames as _L2
    from p6lab.features.microstructure import (
        MICROSTRUCTURE_FEATURE_NAMES as _MICRO,
    )
    _feat_cols = sorted(
        list(_L1.ALL)
        + [c for c in _L2.ALL if c not in ('book_shape_vector', 'weighted_mid')]
        + list(_MICRO)
        + ['fi_fast', 'fi_full']
    )
    relevant = {k: _ds.get(k) for k in (
        'data_file', 'symbol', 'max_snapshots',
        'snapshot_interval_ms', 'num_levels', 'tick_size',
        'start_ms', 'end_ms',
    )}
    relevant['_schema_version'] = SCHEMA_VERSION
    relevant['_feature_cols'] = _feat_cols
    key_str = json.dumps(relevant, sort_keys=True, default=str)
    key_hash = hashlib.sha256(key_str.encode()).hexdigest()[:16]
    cache_dir = Path(_P6LAB) / 'artifacts' / 'cache'
    raw = _ds['data_file']
    if isinstance(raw, (list, tuple)):     stems = [Path(f).stem for f in raw]
    elif isinstance(raw, str) and ',' in raw: stems = [Path(f.strip()).stem for f in raw.split(',')]
    else:                                    stems = [Path(str(raw)).stem]
    human = stems[0] if len(stems) == 1 else f"{stems[0]}+{len(stems)-1}more"
    return cache_dir / f"{_ds['symbol']}_{human}_{key_hash}.parquet"

# ── Cache control (edit per run) ──────────────────────────────────────
# 'auto'          → use cache if present, else build + save  (old behavior)
# 'force_rebuild' → always run slow path; overwrite cache at the end
# 'no_cache'      → always run slow path; DO NOT save (safe to run twice)
# 'force_load'    → require cache hit; raise if missing
CACHE_MODE = 'auto'  # ← edit me
CACHE_FILE_OVERRIDE = Path('<path>' 'NQ_nq-mbo-20260331T0000Z-20260401T0000Z.dbn_b32210693f0ab3d5.parquet')  # e.g. Path('.../cache/NQ_xyz_<hash>.parquet')
cache = Path(CACHE_FILE_OVERRIDE) if CACHE_FILE_OVERRIDE else _cache_path_for(_DS)
cache.parent.mkdir(parents=True, exist_ok=True)
_use_fast = (
    CACHE_MODE == 'force_load'
    or (CACHE_MODE == 'auto' and cache.exists())
)
if CACHE_MODE == 'force_load' and not cache.exists():
    raise FileNotFoundError(f"force_load: {cache} not found")

print(f"cache key: {cache.name}  mode={CACHE_MODE}  path={'FAST' if _use_fast else 'SLOW'}")
if _use_fast:
    # ─── FAST PATH: load all_df from parquet, then reconstruct sidecar dfs ──
    all_df = pd.read_parquet(cache)
    print(f"  loaded from cache: {len(all_df):,} rows")

    # Downstream cells expect `snaps` (only `.timestamp_ms` is accessed) and `n`.
    # Cache-load path re-creates a minimal proxy from __ts_ms__ so those cells
    # work without branching on whether we built fresh or loaded from cache.
    class _TsOnlySnap:
        __slots__ = ('timestamp_ms',)
        def __init__(self, ts): self.timestamp_ms = int(ts)
    snaps = [_TsOnlySnap(t) for t in all_df['__ts_ms__'].to_numpy()]
    n = len(all_df)
    
    # Re-import these only if they aren't already in scope
    from p6lab.features.microstructure import MICROSTRUCTURE_FEATURE_NAMES
    
    # Pull each feature group back out of all_df by column membership
    l1_aligned = all_df[L1FeatureNames.ALL].copy()
    
    # weighted_mid is popped into mid_series (saved as __mid__), never as an all_df column.
    l2_cols = [c for c in L2FeatureNames.ALL if c not in ('book_shape_vector', 'weighted_mid')]
    l2_df   = all_df[l2_cols].copy()
    
    micro_df = all_df[MICROSTRUCTURE_FEATURE_NAMES].copy()
    fi_df    = all_df[['fi_fast', 'fi_full']].copy()
    
    # Sidecars not in the feature dfs:
    mid_series = all_df['__mid__']
    
    # bsvs: the raw 40-dim book shape vectors (NOT the bsv_latent_* encoder outputs)
    bsv_cols = sorted(c for c in all_df.columns 
                      if c.startswith('bsv_') and not c.startswith('bsv_latent_'))
    bsvs_arr = all_df[bsv_cols].to_numpy() if bsv_cols else np.zeros((len(all_df), 0))
    bsvs = [bsvs_arr[i] for i in range(len(bsvs_arr))]   # downstream may expect list form
    
    log.info('01 │ cache hit: all_df=%s  l1=%s  l2=%s  micro=%s  fi=%s  bsv=%d-dim',
             all_df.shape, l1_aligned.shape, l2_df.shape, micro_df.shape, fi_df.shape, bsvs_arr.shape[1])
    # ─── SANITY CHECK (here) ────────────────────────────────
    expected_cols = (set(L1FeatureNames.ALL) 
                     | (set(L2FeatureNames.ALL) - {'book_shape_vector', 'weighted_mid'}) 
                     | set(MICROSTRUCTURE_FEATURE_NAMES) 
                     | {'fi_fast', 'fi_full', '__mid__', '__ts_ms__'})
    missing = expected_cols - set(all_df.columns)
    if missing:
        print(f"  WARN: cache missing {len(missing)} expected cols: {sorted(missing)[:10]}")
        print("    → cache was built with an older feature schema; delete and re-run")
    else:
        print("  cache schema looks good")
else:
    # ─── SLOW PATH: full feature build, then save to cache ───────────
    snaps_all = await collect_snapshots(_DS)
    assert len(snaps_all) > 0, 'No snapshots collected — check _common.NOTEBOOK_DATA_SLICE["data_file"]'
    # Filter to two-sided-book snapshots ONCE, up front. The L2 loop below
    # skips empty-book snaps; if L1 (computed over `snaps`) and the
    # `snaps[:n]` timestamp slice aren't filtered the SAME way, the positional
    # L1 rows / timestamps drift out of alignment with the filtered L2/BSV
    # rows — silently mismatching features, and overflowing a stale-length
    # `valid` mask downstream (IndexError on axis 0). Filtering here makes
    # every downstream array (l1, l2, bsvs, mid, __ts_ms__) describe the SAME
    # rows, so the cold path matches the cache-hit path.
    snaps = []
    _dropped = []   # empty-book diagnostics: (raw_idx, ts_ms, n_trades, n_events)
    for _i, _s in enumerate(snaps_all):
        if _s.bids and _s.asks:
            snaps.append(_s)
        else:
            _dropped.append((
                _i, int(getattr(_s, 'timestamp_ms', 0) or 0),
                len(getattr(_s, 'recent_trades', []) or []),
                len(getattr(_s, 'recent_events', []) or []),
            ))
    log.info('01 │ %d snapshots loaded (%d dropped: empty book)',
             len(snaps), len(_dropped))
    if _dropped:
        _idx = np.array([d[0] for d in _dropped])
        _nt  = np.array([d[2] for d in _dropped])
        _ne  = np.array([d[3] for d in _dropped])
        _N = len(snaps_all)
        # contiguous warmup prefix: how many dropped indices are 0,1,2,...
        _prefix = 0
        for _k, _di in enumerate(_idx):
            if _di == _k:
                _prefix += 1
            else:
                break
        _gaps = np.diff(_idx) if len(_idx) > 1 else np.array([0])
        log.info('01 │   dropped idx [%d..%d]/%d  warmup_prefix=%d  '
                 'in_first_1%%=%d/%d  median_gap=%d  max_gap=%d',
                 int(_idx.min()), int(_idx.max()), _N, _prefix,
                 int((_idx < 0.01*_N).sum()), len(_idx),
                 int(np.median(_gaps)), int(_gaps.max()))
        log.info('01 │   carry trades: %d/%d (%.0f%%)  carry events: %d/%d (%.0f%%)  '
                 'total trades on dropped=%d',
                 int((_nt>0).sum()), len(_dropped), 100*(_nt>0).mean(),
                 int((_ne>0).sum()), len(_dropped), 100*(_ne>0).mean(),
                 int(_nt.sum()))
        log.info('01 │   VERDICT: %s',
                 'WARMUP (contiguous at stream start, drop freely)'
                 if _prefix >= 0.95*len(_idx)
                 else ('SCATTERED w/ trade flow — dropping loses signal'
                       if (_nt>0).mean() > 0.2
                       else 'SCATTERED but ~no trades — safe to drop'))

    #  L1 features 
    adapter = L1Adapter(L1AdapterConfig(tick_size=TICK_SIZE))
    l1_rows = [compute_l1_features(adapter.ingest(s), adapter.history) for s in snaps]
    l1_df = pd.DataFrame(np.array(l1_rows), columns=L1FeatureNames.ALL)
    log.info('01 │ L1 df shape %s', l1_df.shape)

    # L2 features (L2Snapshot uses book_levels: list[(price, bid_sz, ask_sz)])
    _SCALAR_L2_INDICES = [i for i, n in enumerate(L2FeatureNames.ALL)
                          if n != 'book_shape_vector']

    from p6lab.features.vpin import (
        VPINState, VPINConfig, update_vpin_state, compute_vpin,
        classify_trade_lee_ready,
    )
    vpin_state = VPINState()
    vpin_cfg = VPINConfig(avg_daily_volume=300_000.0, bucket_size_fraction=1.0/200,
                          window_size=5)
    _prev_trade_price = 0.0

    from p6lab.features.microstructure import (
        MicrostructureState, update_microstructure, snapshot_features,
        MICROSTRUCTURE_FEATURE_NAMES,
    )
    micro_state = MicrostructureState()

    l2_hist = L2History()
    l2_rows = []
    micro_rows = []
    bsvs = []
    for s in snaps:
        if not (s.bids and s.asks):
            continue
        bid_map = {lvl.price: lvl.volume for lvl in s.bids[:20]}
        ask_map = {lvl.price: lvl.volume for lvl in s.asks[:20]}
        prices = sorted(set(bid_map) | set(ask_map), reverse=True)
        book_levels = [(p, bid_map.get(p, 0.0), ask_map.get(p, 0.0)) for p in prices]
        bp, ap = s.bids[0].price, s.asks[0].price
        bs, as_ = s.bids[0].volume, s.asks[0].volume
        weighted_mid = (bp*as_ + ap*bs) / max(bs+as_, 1e-9)
        plain_mid = 0.5 * (bp + ap)

        trade_dicts = []
        for tr in getattr(s, 'recent_trades', []) or []:
            try:
                tp = float(getattr(tr, 'price', 0.0))
                tv = float(getattr(tr, 'size', 0.0) or getattr(tr, 'volume', 0.0))
                tts = int(getattr(tr, 'timestamp_ms', s.timestamp_ms))
                if tv <= 0:
                    continue
                side = classify_trade_lee_ready(tp, _prev_trade_price, bp, ap)
                update_vpin_state(vpin_state, vpin_cfg, tts, tp, tv, side)
                _prev_trade_price = tp
                trade_dicts.append({'price': tp, 'volume': tv, 'side': side})
            except Exception:
                pass
        vpin_value = compute_vpin(vpin_state, vpin_cfg)
        l2_hist.vpin_value = float(vpin_value) if vpin_value is not None else 0.0

        update_microstructure(micro_state, ts_ms=s.timestamp_ms, mid=plain_mid,
                              trades=trade_dicts)
        micro_rows.append(snapshot_features(micro_state, s.timestamp_ms, plain_mid))

        l2_snap = L2Snapshot(
            timestamp_ms=s.timestamp_ms, symbol=SYMBOL,
            mid_price=plain_mid, book_levels=book_levels,
            recent_events=list(getattr(s, 'recent_events', []) or []),
        )
        l2_hist.append(l2_snap)
        # Compute the 40-dim BSV once and reuse it: pass it into
        # compute_l2_features (for feature [10]'s norm) AND store it, instead
        # of calling compute_book_shape_vector twice per snapshot.
        _bsv = compute_book_shape_vector(l2_snap)
        feat_vec = compute_l2_features(l2_snap, l2_hist, book_shape_vector=_bsv)
        bsvs.append(_bsv)
        row = {L2FeatureNames.ALL[i]: float(feat_vec[i]) for i in _SCALAR_L2_INDICES}
        row['weighted_mid'] = weighted_mid
        l2_rows.append(row)
    l2_df = pd.DataFrame(l2_rows)
    micro_df = pd.DataFrame(micro_rows, columns=MICROSTRUCTURE_FEATURE_NAMES)
    log.info('01 │ L2 df shape %s; micro df shape %s', l2_df.shape, micro_df.shape)

    n = min(len(l1_df), len(l2_df))
    l1_aligned = l1_df.iloc[:n].reset_index(drop=True)
    l2_df      = l2_df.iloc[:n].reset_index(drop=True)
    micro_df   = micro_df.iloc[:n].reset_index(drop=True)

    mid_series = l2_df.pop('weighted_mid')
    log.info('01 │ popped weighted_mid into mid_series — L2 df now %s', l2_df.shape)

    # --- Fragility (vectorized: one NumPy pass via compute_fi_batch, ~100x
    #     faster than the per-row compute_sub_indices loop; output identical).
    #     vpin is the post-loop scalar l2_hist.vpin_value, same as before. ---
    fi = FragilityIndex()
    l1_arr_full = l1_aligned.to_numpy()
    l2_arr_full = l2_df.to_numpy()
    _fi_batch = fi.compute_fi_batch(
        l1_arr_full, l2_arr_full, vpin=l2_hist.vpin_value,
        l2_names=list(l2_df.columns), l1_names=list(l1_aligned.columns),
    )
    fi_df = pd.DataFrame({
        'fi_fast': _fi_batch['fi_fast'],
        'fi_full': _fi_batch['fi_full'],
    })
    log.info('01 │ FI df shape %s', fi_df.shape)

    # Observability invariants and non-zero rates 
    nz_refresh = (l2_df['refresh_rate'] != 0).sum() if 'refresh_rate' in l2_df else 0
    nz_tft = (l2_df['trade_flow_toxicity'] != 0).sum() if 'trade_flow_toxicity' in l2_df else 0
    log.info('01 │ non-zero rates: refresh_rate=%d/%d (%.1f%%), trade_flow_toxicity=%d/%d (%.1f%%)',
             nz_refresh, len(l2_df), 100*nz_refresh/max(len(l2_df),1),
             nz_tft, len(l2_df), 100*nz_tft/max(len(l2_df),1))
    for col in MICROSTRUCTURE_FEATURE_NAMES:
        nz = int((micro_df[col] != 0).sum())
        log.info('01 │ micro %-22s %d/%d (%.1f%%) non-zero',
                 col, nz, len(micro_df), 100*nz/max(len(micro_df),1))
    # Wave 4 Phase 1B VPIN invariant: non-zero TFT must coincide with non-zero refresh_rate
    violations = (l2_df['trade_flow_toxicity'] != 0) & (l2_df['refresh_rate'] == 0)
    if violations.any():
       log.warning('01 │ TFT-without-refresh_rate invariant broken: %d rows', violations.sum())

    # BUILD all_df from components, then save to cache 
    bsvs_arr = np.stack(bsvs[:n]) if bsvs else np.zeros((n, 0))
    bsv_cols = pd.DataFrame(
        bsvs_arr,
        columns=[f'bsv_{i:02d}' for i in range(bsvs_arr.shape[1])],
    )
    all_df = pd.concat([l1_aligned, l2_df, bsv_cols, micro_df, fi_df], axis=1)
    all_df = all_df.loc[:, ~all_df.columns.duplicated()]
    all_df['__ts_ms__'] = [s.timestamp_ms for s in snaps[:n]]
    all_df['__mid__']   = mid_series.values

    assert len(all_df) > 0 and len(all_df.columns) > 5, f"all_df looks wrong: {all_df.shape}"
    all_df.to_parquet(cache)
    print(f"  wrote cache: {len(all_df):,} rows -> {cache}")
    log.info('01 │ all_df shape=%s  cols=%d  cached=%s', all_df.shape, len(all_df.columns), cache.name)
if CACHE_MODE != 'no_cache':
        all_df.to_parquet(cache)
        print(f"  wrote cache: {len(all_df):,} rows -> {cache}")
else:
        print(f"  no_cache: built {len(all_df):,} rows, skipping write")


cache key: NQ_nq-mbo-20260331T0000Z-20260401T0000Z.dbn_b32210693f0ab3d5.parquet  mode=auto  path=FAST
  loaded from cache: 500,000 rows


01 │ cache hit: all_df=(500000, 86)  l1=(500000, 19)  l2=(500000, 16)  micro=(500000, 7)  fi=(500000, 2)  bsv=40-dim


  cache schema looks good
  wrote cache: 500,000 rows -> <path>


# S01 Diagnostic - Features, Cache, Threads

In [4]:
# Wave 4 verification — paste this into a new cell after §01 completes
import numpy as np
from p6lab.features.l2_features import L2_FEATURE_DIM, L2FeatureNames

assert L2_FEATURE_DIM == 18, f"L2_FEATURE_DIM is {L2_FEATURE_DIM}, expected 18"
print(f"L2_FEATURE_DIM = {L2_FEATURE_DIM} ✓")

streak_cols = sorted(c for c in all_df.columns if 'current_streak' in c)
expected = ['current_streak_length', 'current_streak_velocity', 'current_streak_vw_strength']
assert streak_cols == expected, f"streak cols mismatch: {streak_cols}"
print(f"streak cols present: {streak_cols} ✓")

desc = all_df[streak_cols].describe()
print(desc)
for c in streak_cols:
    std = desc.loc['std', c]
    assert std > 0, f"{c} has zero std — detector didn't fire"
    assert np.isfinite(desc.loc[['mean','std','min','max'], c]).all(), f"{c} has non-finite values"
print("✓ all streak features have non-zero std and finite extremes")
print(f"\nall_df.shape = {all_df.shape}")


L2_FEATURE_DIM = 18 ✓
streak cols present: ['current_streak_length', 'current_streak_velocity', 'current_streak_vw_strength'] ✓
       current_streak_length  current_streak_velocity  \
count          ▒.▒▒            ▒.▒▒   
mean                ▒.▒▒                 ▒.▒▒   
std                 ▒.▒▒                 ▒.▒▒   
min                ▒.▒▒                ▒.▒▒   
25%                ▒.▒▒                ▒.▒▒   
50%                 ▒.▒▒                 ▒.▒▒   
75%                 ▒.▒▒                 ▒.▒▒   
max                 ▒.▒▒                 ▒.▒▒   

       current_streak_vw_strength  
count               ▒.▒▒  
mean                     ▒.▒▒  
std                      ▒.▒▒  
min                    ▒.▒▒  
25%                     ▒.▒▒  
50%                      ▒.▒▒  
75%                      ▒.▒▒  
max                     ▒.▒▒  
✓ all streak features have non-zero std and finite extremes

all_df.shape = (500000, 86)


In [5]:
import os
from threadpoolctl import threadpool_info
print(f"=== cache path ===")
print(f"  cache         : {cache}")
print(f"  resolved abs  : {cache.resolve()}")
print(f"  cache.exists(): {cache.exists()}")
print(f"  os.path.isfile: {os.path.isfile(str(cache))}")
print(f"=== variables in scope ===")
for name in ('snaps', 'all_df', 'l1_aligned', 'l2_df', 'micro_df', 'fi_df', 'mid_series'):
    if name in dir():
        v = eval(name)
        if hasattr(v, 'shape'):
            print(f"  {name:12} = {type(v).__name__} shape={v.shape}")
        elif hasattr(v, '__len__'):
            print(f"  {name:12} = {type(v).__name__} len={len(v)}")
        else:
            print(f"  {name:12} = {type(v).__name__}")
    else:
        print(f"  {name:12} = NOT DEFINED")
print("all_df defined?", 'all_df' in dir())
if 'all_df' in dir():
    print(f"shape: {all_df.shape}")
    print("This means feature build DID run. We just need to call to_parquet.")
    all_df.to_parquet(cache)
    print(f"saved -> {cache}")
else:
    print("all_df NOT defined → feature build never ran → else branch is empty")

print("--- threading state ---")
for p in threadpool_info():
    print(f"  {p['user_api']:8} {p['prefix']:20} threads={p['num_threads']}")
raw = os.environ.get("P6LAB_DATA_FILE", "")
specs = [s.strip() for s in raw.split(",")] if raw else []
print(f"Raw P6LAB_DATA_FILE: {raw!r}")
print(f"Profile resolves to {len(specs)} path(s):")
for p in specs:
    exists = pathlib.Path(p).is_file()
    marker = "OK " if exists else "MISSING"
    print(f"  [{marker}] {p}")

print(f"elapsed={time.time()-_t0:.1f}s  rss={_p.memory_info().rss/1e9:.1f}GB")
print(f"snaps loaded: {len(snaps) if 'snaps' in dir() else 'not yet'}")

# Show what's actually in the parent dir(s)
parents = {pathlib.Path(p).parent for p in specs if p}
for parent in parents:
    print(f"\nFiles actually in {parent}:")
    if parent.is_dir():
        for f in sorted(parent.glob("*.dbn.zst")):
            print(f"  {f.name}")
    else:
        print(f"  (parent dir does not exist)")


=== cache path ===
  cache         : <path>
  resolved abs  : <path>
  cache.exists(): True
  os.path.isfile: True
=== variables in scope ===
  snaps        = list len=500000
  all_df       = DataFrame shape=(500000, 86)
  l1_aligned   = DataFrame shape=(500000, 19)
  l2_df        = DataFrame shape=(500000, 16)
  micro_df     = DataFrame shape=(500000, 7)
  fi_df        = DataFrame shape=(500000, 2)
  mid_series   = Series shape=(500000,)
all_df defined? True
shape: (500000, 86)
This means feature build DID run. We just need to call to_parquet.
saved -> <path>
--- threading state ---
  blas     libscipy_openblas    threads=8
  openmp   libgomp              threads=8
  blas     libscipy_openblas    threads=8
  openmp   libgomp              threads=8
Raw P6LAB_DATA_FILE: '<path>'
Profile resolves to 1 path(s):
  [OK ] <path>
elapsed=▒.▒▒s  rss=▒.▒▒GB
snaps loaded: 500000

Files actually in <path>
  nq-mbo-20260323T0000Z-20260324T0000Z.dbn.zst
  nq-mbo-20260324T0000Z-20260325T0000Z.dbn.zs

## Section 02 — L3 Label Assignment

For this research pass we define the L3 label as the **forward 1m direction** (binary up/down). In production this gets replaced by real pattern labels from `library.yaml`. The training framing is the same: predict what the omniscient L3 teacher saw from what the L1/L2 student sees.

In [6]:
from p6lab.validation.labelers import triple_barrier_labels
from p6lab.validation.labelers import mfe_mae_labels

HORIZON_SNAPSHOTS = 600  # 1m at 100ms cadence — used by §04 purge
HORIZON_MS = 60_000

# Wave 9 §H.1.c experiment — MFE/MAE 5-class restricted to clean ±2.
# Clean ±2 = profit barrier hit AND opposite excursion never crossed
# the 1.5-tick stop. We drop {-1, 0, +1}:
#   ±1 → wicky (your stop would have triggered first)
#    0 → timeout / unresolved
# Smaller training set but cleaner feature→label coupling than tb_60s binary.

ts_arr = all_df['__ts_ms__'].to_numpy(dtype=np.int64)
labels_mm = mfe_mae_labels(
    mid_series.to_numpy(), ts_arr,
    horizon_ms=HORIZON_MS,
    up_target_ticks=4.0, down_target_ticks=4.0,
    stop_threshold_ticks=1.5,
    tick_size=TICK_SIZE,
)
y_mm = np.array([lbl.label for lbl in labels_mm], dtype=int)

# Keep only clean ±2. Map +2 → 1, -2 → 0.
valid = np.abs(y_mm) == 2
y = (y_mm[valid] > 0).astype(int)
y_triple = y_mm  # alias in case §03/§04 still reference y_triple

log.info('02 │ MFE/MAE clean-only: -2=%d -1=%d 0=%d +1=%d +2=%d → %d clean (%.1f%% bull)',
         int((y_mm == -2).sum()),
         int((y_mm == -1).sum()),
         int((y_mm ==  0).sum()),
         int((y_mm ==  1).sum()),
         int((y_mm ==  2).sum()),
         int(valid.sum()), 100 * y.mean())



02 │ MFE/MAE clean-only: -2=173552 -1=76694 0=1018 +1=76509 +2=172227 → 345779 clean (▒.▒▒% bull)


## Section 03 — Dataset Construction `X = concat(L1, L2, FI)`

In [7]:
# Explode book_shape_vector (40-d ndarray per row) into 40 scalar columns.
# Without this, pandas stores each BSV as an object-dtype cell and LightGBM
# silently coerces to NaN → 0, wasting 40 dims of signal. See critique §1.3.
bsv_arr_full = np.stack(bsvs[:n])                    # (n, 40)
bsv_cols = pd.DataFrame(
    bsv_arr_full,
    columns=[f'bsv_{i:02d}' for i in range(BOOK_SHAPE_DIM)],
    index=l1_aligned.index,
)

# Wave 4 Phase 3: BSV 1D CNN latent features. When
# artifacts/p6lab/bsv_cnn/encoder_v1.pt exists, load and compute 16-dim
# bottleneck per BSV window. Emits bsv_latent_00..bsv_latent_15 columns.
# At smoketest scale, the encoder is trained on only 2k rows so its latent
# representation is overfit / noisy on the same data — gate behind a
# size threshold same as the micro features.
from p6lab.features.bsv_latent import (
    extract_latent_batch, load_encoder, LATENT_FEATURE_NAMES, BSV_LOOKBACK,
)
_encoder_path = _P6LAB / 'artifacts' / 'p6lab' / 'bsv_cnn' / 'encoder_v1.pt'
_encoder = load_encoder(_encoder_path)

# Wave 4 Phase 2+3: micro + latent features. Both have warmup-window
# instability that hurts AUC on the 15-min sample (demonstrated:
# 0.574 → 0.494 with micro; 0.574 → 0.528 with latent). Gate behind
# a size threshold so they only land at production scale. At ≥10k
# snapshots (16+ minutes of tape per day × 30 days), warmup is a
# small fraction and the features help.
MIN_SNAPS_FOR_L2L3 = 0  # set to 10_000 for smoketest ; 0 for wave 8.5K production scale testing. TBD in real run
include_advanced = len(l2_df) >= MIN_SNAPS_FOR_L2L3
log.info('03 │ include advanced L2/L3 features: %s (n=%d, threshold=%d)',
         include_advanced, len(l2_df), MIN_SNAPS_FOR_L2L3)

latent_cols = None
if _encoder is not None and include_advanced:
    latent_arr = extract_latent_batch(_encoder, bsv_arr_full, lookback=BSV_LOOKBACK)
    latent_cols = pd.DataFrame(
        latent_arr, columns=LATENT_FEATURE_NAMES, index=l1_aligned.index,
    )
    log.info('03 │ BSV latent loaded from %s → %d columns',
             _encoder_path.name, latent_cols.shape[1])
elif _encoder is None:
    log.info('03 │ BSV latent encoder missing at %s — skipping latent cols',
             _encoder_path)
else:
    log.info('03 │ BSV latent encoder available but gated off at smoketest scale')

parts = [
    l1_aligned.reset_index(drop=True),
    l2_df.reset_index(drop=True),
    bsv_cols.reset_index(drop=True),
]
if include_advanced:
    parts.append(micro_df.reset_index(drop=True))
if latent_cols is not None:
    parts.append(latent_cols.reset_index(drop=True))
parts.append(fi_df.reset_index(drop=True))

X_full = pd.concat(parts, axis=1)
# Drop constant/duplicate columns
constants = [c for c in X_full.columns if X_full[c].std() == 0]
X = X_full.drop(columns=constants)
# De-dup columns (l1 spread_bps and l2 spread_bps would overlap)
X = X.loc[:, ~X.columns.duplicated()]
log.info('03 │ X shape %s (dropped %d constants; +40 bsv%s%s)',
         X.shape, len(constants),
         ' + 7 micro' if include_advanced else '',
         ' + 16 latent' if latent_cols is not None else '')


03 │ include advanced L2/L3 features: True (n=500000, threshold=0)
03 │ BSV latent loaded from encoder_v1.pt → 16 columns
03 │ X shape (500000, 100) (dropped 0 constants; +40 bsv + 7 micro + 16 latent)


## Section 03b — Wave 9 A2d: Feature-Population Audit

Surface dead, sparse, or NaN/Inf features in `X` before §04 training. 
Sentinel check on Wave 9 A2a momentum features (`signed_flow_60s`, 
`imbalance_velocity_5s`, `trade_streak`, `liquidity_withdrawal_asym`).

References: 
[ELI5 Footnote A](../../reports/P6LAB-ELI5-LABELS-AND-STRATEGY.md), 
[Build doc §H.5 A2d row](../../reports/P6LAB-WAVE-9-10-BUILD-PHASES.md).


In [8]:
# §03b — Wave 9 A2d: Feature-Population Audit
#
# Catches features that look healthy in the schema but are silently dead
# (constant), sparse (>95% zero), or contaminated with NaN/Inf. The §03
# step already drops std==0 columns; this audit catches truly dead
# (constant: <=1 unique finite value — SCALE-AWARE, so tiny-but-varying
# features like realized_variance_30s ~1e-7 are NOT false-flagged) and
# surfaces a presence check on the new
# Wave 9 A2a momentum features so we know they're carrying signal at
# this data scale.

import numpy as np

# A2a momentum features — must be alive in X for the directional
# thesis test to be meaningful.
A2A_MOMENTUM_FEATURES = (
    "signed_flow_60s",
    "imbalance_velocity_5s",
    # a23f24f replaced unbounded 'trade_streak' with the three cup_flip
    # StreakDetector features — audit the current names.
    "current_streak_length",
    "current_streak_velocity",
    "current_streak_vw_strength",
    "liquidity_withdrawal_asym",
)

# DEAD = constant (<=1 unique finite value). Scale-aware on purpose: an
# absolute std<1e-6 test falsely flagged realized_variance_30s (std~2.5e-7
# yet 465k unique values — alive). nunique catches genuine deadness only.
SPARSE_NZ = 0.05

print(f"Feature matrix X: {X.shape[0]} rows × {X.shape[1]} columns")
print()

flags = {"DEAD": 0, "SPARSE": 0, "NAN": 0, "INF": 0, "OK": 0}
problem_rows: list[str] = []
header = (
    f"{'feature':<36} {'mean':>12} {'std':>12} {'nz%':>7} "
    f"{'nan':>5} {'inf':>5}  flag"
)

for col in X.columns:
    vec = X[col]
    mean_v = float(vec.mean())
    std_v = float(vec.std())
    nunique = int(vec.nunique(dropna=True))
    nz = float((vec != 0).mean())
    n_nan = int(vec.isna().sum())
    inf_mask = np.isinf(vec.fillna(0).to_numpy())
    n_inf = int(inf_mask.sum())

    flag_parts = []
    if nunique <= 1:
        flag_parts.append("DEAD")
        flags["DEAD"] += 1
    elif nz < SPARSE_NZ:
        flag_parts.append("sparse")
        flags["SPARSE"] += 1
    if n_nan > 0:
        flag_parts.append(f"nan={n_nan}")
        flags["NAN"] += 1
    if n_inf > 0:
        flag_parts.append(f"inf={n_inf}")
        flags["INF"] += 1
    flag_str = " ".join(flag_parts)
    if not flag_str:
        flags["OK"] += 1

    # Only print rows that need attention OR are A2a sentinel features
    if flag_str or col in A2A_MOMENTUM_FEATURES:
        problem_rows.append(
            f"{col:<36} {mean_v:>12.4f} {std_v:>12.4f} {nz:>6.1%} "
            f"{n_nan:>5d} {n_inf:>5d}  {flag_str}"
        )

if problem_rows:
    print(header)
    print("-" * 92)
    for line in problem_rows:
        print(line)
    print()
print(
    f"Summary: {flags['OK']} healthy / "
    f"{flags['DEAD']} dead / {flags['SPARSE']} sparse / "
    f"{flags['NAN']} with NaN / {flags['INF']} with Inf"
)

# ── Wave 9 A2a presence check ────────────────────────────────────────────
print()
print("Wave 9 A2a momentum features — presence in X:")
for name in A2A_MOMENTUM_FEATURES:
    if name in X.columns:
        std_v = float(X[name].std())
        nz = float((X[name] != 0).mean())
        if int(X[name].nunique(dropna=True)) <= 1:
            verdict = "DEAD (constant — upstream wiring broken?)"
        elif nz < SPARSE_NZ:
            verdict = f"sparse — only {nz:.1%} nonzero"
        else:
            verdict = "alive"
        print(f"  {name:<32} std={std_v:.4f}  nz={nz:>5.1%}  [{verdict}]")
    else:
        print(f"  {name:<32} [MISSING — not threaded into X]")


Feature matrix X: 500000 rows × 100 columns

feature                                      mean          std     nz%   nan   inf  flag
--------------------------------------------------------------------------------------------
signed_flow_60s                           ▒.▒▒       ▒.▒▒  ▒.▒▒%     0     0  
imbalance_velocity_5s                     ▒.▒▒       ▒.▒▒  ▒.▒▒%     0     0  
current_streak_length                      ▒.▒▒       ▒.▒▒  ▒.▒▒%     0     0  
liquidity_withdrawal_asym                 ▒.▒▒       ▒.▒▒  ▒.▒▒%     0     0  
current_streak_velocity                    ▒.▒▒       ▒.▒▒  ▒.▒▒%     0     0  
current_streak_vw_strength                 ▒.▒▒       ▒.▒▒  ▒.▒▒%     0     0  

Summary: 100 healthy / 0 dead / 0 sparse / 0 with NaN / 0 with Inf

Wave 9 A2a momentum features — presence in X:
  signed_flow_60s                  std=▒.▒▒  nz=▒.▒▒%  [alive]
  imbalance_velocity_5s            std=▒.▒▒  nz=▒.▒▒%  [alive]
  current_streak_length            std=▒.▒▒  nz=▒.▒▒%  

In [9]:
# §03c-fast — TB + MM labels only, no matcher loop, no pf_* columns.
# Use when you only need the TB/MFE-MAE columns: this skips the matcher
# entirely so multi_labels lacks pf_*, which §03d / §04 don't need for the
# MFE/MAE experiment.
#
# NOTE: the old "skips the 10-min loop" rationale is obsolete. The cost was
# the pure-Python barrier walks in compute_label_set, not the matcher; those
# are now numba-JIT'd (labelers._tb_kernel / _mm_kernel), so the full §03c
# also runs in seconds. This cell now only saves the (already cheap) matcher
# pass + pf_* columns.

import numpy as np
import pandas as pd
from p6lab.validation.labelers import LabelSpec, compute_label_set

HORIZONS_MS = (60_000, 120_000, 180_000, 300_000)
ts_arr_all = np.asarray([s.timestamp_ms for s in snaps[:n]], dtype=np.int64)
mid_full = mid_series.to_numpy()

tb_mfe_specs = []
for h_ms in HORIZONS_MS:
    h_name = f"{h_ms // 1000}s"
    tb_mfe_specs.append(LabelSpec(
        f"tb_{h_name}", kind="tb",
        horizon_ms=h_ms,
        up_target_ticks=4.0, down_target_ticks=4.0, tick_size=TICK_SIZE,
    ))
    tb_mfe_specs.append(LabelSpec(
        f"mm_{h_name}", kind="mfe_mae",
        horizon_ms=h_ms,
        up_target_ticks=4.0, down_target_ticks=4.0,
        stop_threshold_ticks=1.5, tick_size=TICK_SIZE,
    ))

tb_mfe_dict = compute_label_set(mid_full, ts_arr_all, tb_mfe_specs)
multi_labels = pd.DataFrame(tb_mfe_dict)

print(f"multi_labels (TB + MM only): {multi_labels.shape}  columns={list(multi_labels.columns)}")
print()
print("Class distribution per column:")
for col in multi_labels.columns:
    vc = multi_labels[col].value_counts().sort_index()
    bins = "  ".join(f"{int(k):+d}={v}" for k, v in vc.items())
    print(f"  {col:<22} {bins}")


multi_labels (TB + MM only): (500000, 8)  columns=['tb_60s', 'mm_60s', 'tb_120s', 'mm_120s', 'tb_180s', 'mm_180s', 'tb_300s', 'mm_300s']

Class distribution per column:
  tb_60s                 -1=250246  +0=1008  +1=248736
  mm_60s                 -2=173552  -1=76694  +0=1008  +1=76509  +2=172227
  tb_120s                -1=250300  +0=933  +1=248757
  mm_120s                -2=173606  -1=76694  +0=933  +1=76509  +2=172248
  tb_180s                -1=250347  +0=867  +1=248776
  mm_180s                -2=173653  -1=76694  +0=867  +1=76509  +2=172267
  tb_300s                -1=250505  +0=648  +1=248837
  mm_300s                -2=173811  -1=76694  +0=648  +1=76509  +2=172328


## Section 03c — multi-target label dataframe (Wave 9/10)

Builds the canonical 16-column label set: TB × {60s,120s,180s,300s} + 
MFE/MAE × {60s,120s,180s,300s} + pattern_fire_binary × {60s,120s,180s,300s} + 
pattern_fire_best × {60s,120s,180s,300s}. §04 will train one model per 
column and emit a side-by-side §04d/§04e diagnostic to identify which 
(label_kind, horizon) the features actually answer.

Pattern-firing labels require a model registry with previously-mined 
templates. **Cold start (no registry):** pattern_fire columns are emitted 
but all-zero — re-run NB06 once through §11b/§12 to mine templates, then 
re-run §03c with the warm registry.

References: 
[ELI5 §9.3 5-class label](../../reports/P6LAB-ELI5-LABELS-AND-STRATEGY.md), 
[Build doc §H.1.b multi-target labels](../../reports/P6LAB-WAVE-9-10-BUILD-PHASES.md), 
Wave 10-A pattern_firing_labels.


In [10]:
# §03c — Wave 9+10 multi-target label dataframe
#
# Constructs the canonical 16-column label set used by the §04 multi-spec
# training loop. Three label families × four horizons:
#   TB              (triple_barrier_labels)         — 4 cols
#   MFE/MAE         (mfe_mae_labels, 5-class)        — 4 cols
#   PF binary       (pattern_firing_labels, binary) — 4 cols
#   PF best         (pattern_firing_labels, best)   — 4 cols
#
# Pattern-firing requires templates. Resolution order:
#   1. Registry templates (CURRENT.json) if centroid dim matches the live
#      L2 frame.
#   2. Inline re-mine using NB06 §11b's L2-only rules, when the registry is
#      missing OR its centroids are stale (feature-set drift since promotion).
#   3. All-zero pf columns when re-mine yields nothing.

import json
import pickle

from p6lab.validation.labelers import (
    LabelSpec, compute_label_set, pattern_firing_labels,
)
from p6lab.patterns.template_matcher import (
    MatchContext, PatternTemplate, TemplateMatcher,
)

HORIZONS_MS = (60_000, 120_000, 180_000, 300_000)
PF_STRIDE = 5  # match every Nth snapshot for speed; reduce to 1 for max density
# Firing threshold for the pf_* labels. The live engine treats ensemble score
# >= 0.85 as a tier-A (high-confidence) match. The old 0.60 admitted weak
# tier-C matches on almost every snapshot, saturating pf_*_binary to ~99.7%
# positive — so §04 CPCV folds had a single class ("no usable folds") and the
# 7-class pf_*_best AUC came out NaN. 0.85 makes "a pattern fired" mean an
# actionable fire. If pf_* then ends up too sparse to train, lower toward 0.72
# (tier-B).
PF_MIN_SCORE = 0.85

# Live matcher frame = l2_df minus book_shape_vector (weighted_mid is already
# popped in §01). Centroids and global_covariance MUST match this width.
from p6lab.features.l2_features import L2_MATCH_EXCLUDE
# Single source of truth — keeps this mining frame identical to the live
# engine's (engine._latest_l2_features), so centroid dims can't diverge.
_L2_MATCH_EXCLUDE = set(L2_MATCH_EXCLUDE)
_match_cols = [c for c in l2_df.columns if c not in _L2_MATCH_EXCLUDE]
_expected_centroid_dim = len(_match_cols)
# ── 1. Try registry (warm start) ──────────────────────────────────────────
_REGISTRY = MODELS_DIR / "CURRENT.json"
matcher_for_pf: TemplateMatcher | None = None
if _REGISTRY.is_file():
    try:
        reg = json.loads(_REGISTRY.read_text())
        model_path = (_REGISTRY.parent / reg["model_path"]).resolve()
        with open(model_path, "rb") as fh:
            md = pickle.load(fh)
        templates_md = md.get("matcher_templates") or {}
        centroids_md = md.get("matcher_centroids") or {}
        contexts_md = md.get("pattern_contexts") or {}
        covariance_md = md.get("global_covariance")
        if templates_md:
            stale_dims = {
                pid: int(np.asarray(centroids_md.get(pid, [])).shape[0])
                for pid in templates_md
            }
            if any(d != _expected_centroid_dim for d in stale_dims.values()):
                log.warning(
                    "03c │ registry centroid dims %s != live %d "
                    "(feature-set drift since %s promoted) — falling back "
                    "to inline re-mine",
                    stale_dims, _expected_centroid_dim, model_path.name,
                )
            else:
                matcher_for_pf = TemplateMatcher()
                matcher_for_pf.templates = {
                    pid: PatternTemplate(
                        pattern_id=pid,
                        book_series=np.asarray(tpl),
                        feature_centroid=np.asarray(centroids_md[pid]),
                        pattern_context=contexts_md.get(pid, {}),
                    )
                    for pid, tpl in templates_md.items()
                }
                if covariance_md is not None:
                    cov_arr = np.asarray(covariance_md)
                    if cov_arr.shape == (_expected_centroid_dim,
                                         _expected_centroid_dim):
                        matcher_for_pf.fit_covariance(cov_arr)
                log.info(
                    "03c │ loaded %d templates from %s",
                    len(matcher_for_pf.templates), model_path.name,
                )
    except Exception as exc:
        log.warning(
            "03c │ failed to load registry templates (%s); will re-mine inline",
            exc,
        )
        matcher_for_pf = None
else:
    log.warning(
        "03c │ no registry at %s — will re-mine inline", _REGISTRY,
    )

# ── 2. Inline re-mine fallback ────────────────────────────────────────────
# Mirrors NB06 §11b's rule-based mining, restricted to L2-only rules (the
# bsv_*-based `lgbm_tier_a_fwd_up` rule depends on §02's exploded BSV
# columns and isn't trivially re-derivable here). Produces templates with
# the *current* feature dim, so subsequent match_all() can't broadcast-fail.
if matcher_for_pf is None:
    _MINE_TEMPLATE_ROWS = 10
    _bsv_stack = np.stack(bsvs[:n])  # (n, 40)
    _rules_inline: dict[str, np.ndarray] = {
        "imbalance_ema_surge": (l2_df["imbalance_ema"].abs().to_numpy() > 0.3),
        "depth_ratio_skew":    (l2_df["depth_ratio"].abs().to_numpy() > 1.5),
    }
    mined: dict[str, PatternTemplate] = {}
    for pat_id, mask in _rules_inline.items():
        hits = np.where(mask)[0]
        windows = [
            _bsv_stack[idx - _MINE_TEMPLATE_ROWS : idx]
            for idx in hits if idx >= _MINE_TEMPLATE_ROWS
        ]
        if not windows:
            log.info(
                "03c │ inline re-mine %-24s 0 usable windows, skipped", pat_id,
            )
            continue
        book_series = np.median(np.stack(windows), axis=0)  # (10, 40)
        centroid = l2_df[_match_cols].iloc[hits].median().to_numpy(dtype=float)
        mined[pat_id] = PatternTemplate(
            pattern_id=pat_id,
            book_series=book_series,
            feature_centroid=centroid,
            pattern_context={"vix_regime": "normal"},
        )
        log.info(
            "03c │ inline re-mine %-24s %5d firings, centroid_dim=%d",
            pat_id, len(hits), centroid.shape[0],
        )
    if mined:
        matcher_for_pf = TemplateMatcher()
        matcher_for_pf.templates = mined
        log.info("03c │ inline re-mined %d templates", len(mined))
    else:
        log.warning(
            "03c │ inline re-mine produced 0 templates — pf cols will be zero",
        )

# ── 3. Run matcher in batch to collect firings (skipped if no templates) ──
firings: list[tuple[int, str, float]] = []
if matcher_for_pf is not None and matcher_for_pf.templates:
    BSV_WINDOW_LEN = 30
    SYMBOL = NOTEBOOK_DATA_SLICE.get("symbol", "NQ")

    # Hoist per-iteration work out of the loop: `l2_df[_match_cols]` rebuilt a
    # full-frame copy every iteration (~n/stride times), and `np.stack` re-
    # allocated each BSV window. Materialize both once; index/slice rows below.
    _match_arr = l2_df[_match_cols].to_numpy(dtype=float)
    _bsv_all = np.stack(bsvs[:n])

    n_matched = 0
    for i in range(BSV_WINDOW_LEN, n, PF_STRIDE):
        bsv_window = _bsv_all[i - BSV_WINDOW_LEN + 1 : i + 1]
        feat_now = _match_arr[i]
        ts = int(snaps[i].timestamp_ms)
        ctx = MatchContext(
            time_of_day_minutes=int((ts // 60_000) % 1440),
            vix_level=18.0, vix_regime="normal",
            relative_volume=1.0, instrument=SYMBOL,
        )
        results = matcher_for_pf.match_all(
            current_book_series=bsv_window,
            current_features=feat_now,
            templates=None,  # use stored
            context=ctx,
            min_score=PF_MIN_SCORE,
        )
        for r in results:
            firings.append((i, r.pattern_id, float(r.ensemble_score)))
        n_matched += 1
    log.info(
        "03c │ matched %d snapshots (stride=%d), collected %d firings",
        n_matched, PF_STRIDE, len(firings),
    )

# ── Build TB + MFE/MAE specs (4 horizons each) ─────────────────────────────
ts_arr_all = np.asarray([s.timestamp_ms for s in snaps[:n]], dtype=np.int64)
mid_full = mid_series.to_numpy()

tb_mfe_specs: list[LabelSpec] = []
for h_ms in HORIZONS_MS:
    h_name = f"{h_ms // 1000}s"
    tb_mfe_specs.append(LabelSpec(
        f"tb_{h_name}", kind="tb",
        horizon_ms=h_ms,
        up_target_ticks=4.0, down_target_ticks=4.0, tick_size=TICK_SIZE,
    ))
    tb_mfe_specs.append(LabelSpec(
        f"mm_{h_name}", kind="mfe_mae",
        horizon_ms=h_ms,
        up_target_ticks=4.0, down_target_ticks=4.0,
        stop_threshold_ticks=1.5, tick_size=TICK_SIZE,
    ))

tb_mfe_dict = compute_label_set(mid_full, ts_arr_all, tb_mfe_specs)

# ── Build pattern_firing labels (zero-filled if no firings) ────────────────
pf_dict: dict[str, np.ndarray] = {}
for h_ms in HORIZONS_MS:
    h_name = f"{h_ms // 1000}s"
    pf_dict[f"pf_{h_name}_binary"], _ = pattern_firing_labels(
        ts_arr_all, firings, horizon_ms=h_ms, encoding="binary",
    )
    pf_dict[f"pf_{h_name}_best"], _ = pattern_firing_labels(
        ts_arr_all, firings, horizon_ms=h_ms, encoding="best",
    )

# ── Combine into one dataframe + summary ───────────────────────────────────
multi_labels = pd.DataFrame({**tb_mfe_dict, **pf_dict})
log.info("03c │ multi_labels shape=%s columns=%d",
         multi_labels.shape, multi_labels.shape[1])

print(f"\nMulti-target label dataframe: {multi_labels.shape}")
print(f"Columns: {list(multi_labels.columns)}")
print()
print("Class distribution per column:")
for col in multi_labels.columns:
    vc = multi_labels[col].value_counts().sort_index()
    bins = "  ".join(f"{int(k):+d}={v}" for k, v in vc.items())
    print(f"  {col:<22} {bins}")


03c │ loaded 6 templates from v1_nq_fwd1m.pkl


03c │ matched 99994 snapshots (stride=5), collected 0 firings
03c │ multi_labels shape=(500000, 16) columns=16



Multi-target label dataframe: (500000, 16)
Columns: ['tb_60s', 'mm_60s', 'tb_120s', 'mm_120s', 'tb_180s', 'mm_180s', 'tb_300s', 'mm_300s', 'pf_60s_binary', 'pf_60s_best', 'pf_120s_binary', 'pf_120s_best', 'pf_180s_binary', 'pf_180s_best', 'pf_300s_binary', 'pf_300s_best']

Class distribution per column:
  tb_60s                 -1=250246  +0=1008  +1=248736
  mm_60s                 -2=173552  -1=76694  +0=1008  +1=76509  +2=172227
  tb_120s                -1=250300  +0=933  +1=248757
  mm_120s                -2=173606  -1=76694  +0=933  +1=76509  +2=172248
  tb_180s                -1=250347  +0=867  +1=248776
  mm_180s                -2=173653  -1=76694  +0=867  +1=76509  +2=172267
  tb_300s                -1=250505  +0=648  +1=248837
  mm_300s                -2=173811  -1=76694  +0=648  +1=76509  +2=172328
  pf_60s_binary          +0=499433
  pf_60s_best            +0=499433
  pf_120s_binary         +0=498879
  pf_120s_best           +0=498879
  pf_180s_binary         +0=498317
  pf_

## Section 03d — Label-quality audit

Before deciding whether to switch from binary `y` (§02) to MFE/MAE 5-class
(`mm_60s`), check whether the current label is even learnable from the
current features. Five no-train checks (~30s runtime):

1. **Class balance** of current `y` + intra-day drift across 8 row-segments.
2. **Mutual information**: top 12 features vs `y` — if top MI ≈ 0, the
   label is uncorrelated with anything we feed the model.
3. **TB horizon sensitivity**: class balance + timeout fraction at 60/120/180/300s.
4. **MFE/MAE actionable fraction**: how big is the ±2 "clean move" subset per horizon.
5. **Entropy** comparison: binary vs TB-3-class vs MM-5-class — quantifies what
   the binary label throws away.

Final verdict block recommends switch / tune horizon / look elsewhere.

In [11]:
# §03d — Label-quality audit
#
# Pre-flight before Wave 9 §H.1.c (binary → MFE/MAE switch). Five no-train checks on the current binary y
# (from §02) and the multi-horizon label set (from §03c). Verdict at the end.

from sklearn.feature_selection import mutual_info_classif

print("=" * 72)
print(" Label-quality audit")
print("=" * 72)

# [1] current binary y from §02
print(f"\n[1] Current binary y (tb_60s sign, timeouts dropped):")
print(f"    n = {len(y):,}  pos% = {100*y.mean():.2f}")

ts_valid = ts_arr_all[valid]
seg_idx = np.linspace(0, len(y), 9, dtype=int)
print(f"    intra-day drift (8 row-segments):")
for i in range(8):
    a, b = seg_idx[i], seg_idx[i+1]
    if b <= a:
        continue
    seg_pos = y[a:b].mean()
    t_a = (ts_valid[a] // 60_000) % (60*24)
    t_b = (ts_valid[b-1] // 60_000) % (60*24)
    print(f"      seg{i+1}: rows [{a:>7}..{b:>7}]  "
          f"~{t_a//60:02d}:{t_a%60:02d}-{t_b//60:02d}:{t_b%60:02d}  "
          f"pos%={100*seg_pos:.2f}")
drift = y[seg_idx[0]:seg_idx[1]].mean() - y[seg_idx[-2]:seg_idx[-1]].mean()
print(f"    first-vs-last drift = {100*drift:+.2f} pts")

# [2] label-feature MI on a 30k subsample
print(f"\n[2] MI: top 12 features vs current y")
rng = np.random.default_rng(42)
X_valid = X.loc[valid].reset_index(drop=True)
n_sub = min(30_000, len(y))
sub_idx = rng.choice(len(y), n_sub, replace=False)
X_sub = X_valid.iloc[sub_idx].fillna(0.0).replace([np.inf, -np.inf], 0.0)
y_sub = y[sub_idx]
mi = mutual_info_classif(X_sub.to_numpy(), y_sub, random_state=42)
mi_ranked = sorted(zip(X_sub.columns, mi), key=lambda t: -t[1])[:12]
for name, score in mi_ranked:
    print(f"    {name:<30} MI = {score:.4f}")
mi_top1 = float(mi_ranked[0][1]) if mi_ranked else 0.0
mi_median = float(np.median(mi))
print(f"    top MI = {mi_top1:.4f}   median MI = {mi_median:.4f}")

# [3] TB horizon sensitivity
print(f"\n[3] TB horizon sensitivity:")
print(f"    {'col':<10} {'n_up':>8} {'n_dn':>8} {'n_to':>8}  pos%(no-to)   to%")
for h in (60, 120, 180, 300):
    col = f"tb_{h}s"
    s = multi_labels[col]
    n_up = int((s == 1).sum())
    n_dn = int((s == -1).sum())
    n_to = int((s == 0).sum())
    n_res = n_up + n_dn
    pos_pct = 100 * n_up / n_res if n_res else 0.0
    to_pct = 100 * n_to / len(s)
    print(f"    {col:<10} {n_up:>8} {n_dn:>8} {n_to:>8}    {pos_pct:>6.2f}    {to_pct:>5.2f}")

# [4] MFE/MAE actionable fraction per horizon
print(f"\n[4] MFE/MAE actionable population (±2):")
print(f"    {'col':<10} {'-2':>8} {'-1':>8} {'+0':>8} {'+1':>8} {'+2':>8}  actionable%")
for h in (60, 120, 180, 300):
    col = f"mm_{h}s"
    s = multi_labels[col]
    counts = {k: int((s == k).sum()) for k in (-2, -1, 0, 1, 2)}
    n = sum(counts.values()) or 1
    act = (counts[-2] + counts[2]) / n * 100
    print(f"    {col:<10} {counts[-2]:>8} {counts[-1]:>8} {counts[0]:>8} "
          f"{counts[1]:>8} {counts[2]:>8}    {act:>6.2f}")

# [5] entropy comparison
print(f"\n[5] Information content (entropy, bits):")
def _entropy(s):
    vc = pd.Series(s).value_counts(normalize=True).to_numpy()
    vc = vc[vc > 0]
    return float(-(vc * np.log2(vc)).sum())
print(f"    binary y (current):         {_entropy(y):.3f} bits   max=1.000")
print(f"    tb_60s (3-class):           {_entropy(multi_labels['tb_60s']):.3f} bits   max=1.585")
print(f"    mm_60s (5-class):           {_entropy(multi_labels['mm_60s']):.3f} bits   max=2.322")

# ── Verdict ──────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print(" VERDICT")
print("=" * 72)
to_60 = float((multi_labels['tb_60s'] == 0).mean())
mm_60_act = float((multi_labels['mm_60s'].abs() == 2).mean())
print(f"  • Intra-day drift = {100*drift:+.2f} pts "
      f"{'(non-stationary)' if abs(100*drift) > 5 else '(stationary)'}")
print(f"  • Top MI = {mi_top1:.4f} "
      f"{'(near-zero — label/feature mismatch)' if mi_top1 < 0.005 else '(some signal)'}")
print(f"  • TB 60s timeout = {100*to_60:.2f}% (binary discards this fraction)")
print(f"  • MM 60s actionable (±2) = {100*mm_60_act:.2f}%")
print()
print(" Switch decision (Wave 9 §H.1.c — directional model target):")
if mi_top1 < 0.005 and mm_60_act > 0.40:
    print("    → SWITCH to mm_60s with class-weighted training. Binary throws away")
    print("      cleanness signal; ±2 subset is large enough to train on.")
elif mi_top1 < 0.005:
    print("    → Tune horizon (try 120s/180s) or cost_thresholded_binary —")
    print("      label noise at 60s is the bottleneck.")
else:
    print("    → Label is fine — bottleneck is elsewhere (features or sample size).")

 Label-quality audit

[1] Current binary y (tb_60s sign, timeouts dropped):
    n = 345,779  pos% = ▒.▒▒
    intra-day drift (8 row-segments):
      seg1: rows [      0..  43222]  ~21:55-02:15  pos%=▒.▒▒
      seg2: rows [  43222..  86444]  ~02:15-05:35  pos%=▒.▒▒
      seg3: rows [  86444.. 129667]  ~05:35-08:40  pos%=▒.▒▒
      seg4: rows [ 129667.. 172889]  ~08:40-11:52  pos%=▒.▒▒
      seg5: rows [ 172889.. 216111]  ~11:52-14:10  pos%=▒.▒▒
      seg6: rows [ 216111.. 259334]  ~14:10-16:03  pos%=▒.▒▒
      seg7: rows [ 259334.. 302556]  ~16:03-17:52  pos%=▒.▒▒
      seg8: rows [ 302556.. 345779]  ~17:52-19:55  pos%=▒.▒▒
    first-vs-last drift = ▒.▒▒ pts

[2] MI: top 12 features vs current y
    current_streak_velocity        MI = ▒.▒▒
    l1_shape_bid_unit              MI = ▒.▒▒
    l1_shape_ask_unit              MI = ▒.▒▒
    l1_shape_vector                MI = ▒.▒▒
    spread_bps_l1                  MI = ▒.▒▒
    spread_bps                     MI = ▒.▒▒
    trade_flow_toxicity   

## Section 03e — Pattern-firing label audit

`§03c` already emits 8 `pf_*` columns into `multi_labels`:
`pf_{60,120,180,300}s × {binary, best}`. This cell answers: are these
usable as a primary training target (Strategy B / Wave 10-A), or do they
need sharper mining rules first?

**Source caveat.** §03c's inline re-mine path was used because the
registry centroids are stale (dim 14 vs live 16). Inline re-mine fires
only L2-only rules — `|imbalance_ema| > 0.3` and `|depth_ratio| > 1.5`
— and skips the BSV-based `lgbm_tier_a_fwd_up` rule. So these `pf_*`
labels are built from 2 of the 3 live templates, with rules loose
enough to fire on almost every row.

In [12]:
# §03e — Pattern-firing label audit
#
# Assess whether multi_labels['pf_*'] is usable as a Strategy B training
# target (Wave 10-A), or whether the inline re-mine path produces a
# degenerate label that can't be learned regardless of model class.
#
# NaN handling: pattern_firing_labels emits NaN for rows near end-of-data
# (tail can't see horizon_ms ahead). Drop those before MI / balance stats.

from sklearn.feature_selection import mutual_info_classif

print("=" * 72)
print(" Pattern-firing label audit (pf_*)")
print("=" * 72)

# [1] Class balance (drop NaN so percentages sum to 100)
print(f"\n[1] Class balance per pf_* column (NaN dropped):")
print(f"    {'col':<22}  details" + " " * 38 + "flag")
for col in [c for c in multi_labels.columns if c.startswith('pf_')]:
    s = multi_labels[col].dropna()
    n = len(s)
    if n == 0:
        print(f"    {col:<22}  [all NaN]")
        continue
    if 'binary' in col:
        pos = float((s == 1).mean())
        imbal = abs(pos - 0.5) * 2
        flag = 'SEVERE' if imbal > 0.85 else ('MODERATE' if imbal > 0.6 else 'OK')
        details = f"neg%={100*(1-pos):>6.2f}  pos%={100*pos:>6.2f}  imb={imbal:>.3f}"
        print(f"    {col:<22}  {details:<48}{flag}")
    else:
        cnts = {k: int((s == k).sum()) for k in (0, 1, 2)}
        worst = max(cnts.values()) / n
        flag = 'SEVERE' if worst > 0.95 else ('MODERATE' if worst > 0.8 else 'OK')
        details = (f"0%={100*cnts[0]/n:>5.2f} 1%={100*cnts[1]/n:>5.2f} "
                   f"2%={100*cnts[2]/n:>5.2f} worst={100*worst:>5.1f}%")
        print(f"    {col:<22}  {details:<48}{flag}")

# [2] MI: features vs pf_60s_binary (filter to finite rows first)
print(f"\n[2] MI: top 8 features vs pf_60s_binary")
rng = np.random.default_rng(42)
pf_full = multi_labels['pf_60s_binary'].to_numpy()
finite_idx = np.where(~np.isnan(pf_full))[0]
n_dropped = len(pf_full) - len(finite_idx)
if n_dropped:
    print(f"    (dropped {n_dropped} NaN rows from pf_60s_binary)")

pf_mi_top1 = 0.0
if len(finite_idx) == 0:
    print(f"    pf_60s_binary fully NaN — skipping MI")
else:
    n_sub = min(30_000, len(finite_idx))
    sub_idx = rng.choice(finite_idx, n_sub, replace=False)
    X_sub = X.iloc[sub_idx].fillna(0.0).replace([np.inf, -np.inf], 0.0)
    y_pf = pf_full[sub_idx].astype(int)
    if y_pf.std() < 1e-9:
        print(f"    pf_60s_binary degenerate "
              f"(pos rate {(y_pf==1).mean():.4f}) — MI undefined")
    else:
        mi_pf = mutual_info_classif(X_sub.to_numpy(), y_pf, random_state=42)
        pf_ranked = sorted(zip(X_sub.columns, mi_pf), key=lambda t: -t[1])[:8]
        for name, score in pf_ranked:
            print(f"    {name:<30} MI = {score:.4f}")
        pf_mi_top1 = float(pf_ranked[0][1])
        print(f"    top MI = {pf_mi_top1:.4f}")

# [3] Per-segment firing rate drift (pandas .mean() skips NaN by default)
print(f"\n[3] Intra-day firing rate (pf_60s_binary):")
ts_all = ts_arr_all[:len(multi_labels)]
seg_idx_pf = np.linspace(0, len(multi_labels), 6, dtype=int)
seg_rates = []
for i in range(5):
    a, b = seg_idx_pf[i], seg_idx_pf[i+1]
    if b <= a:
        continue
    seg_rate = float(multi_labels['pf_60s_binary'].iloc[a:b].mean())
    seg_rates.append(seg_rate)
    t_a = (ts_all[a] // 60_000) % (60 * 24)
    t_b = (ts_all[b-1] // 60_000) % (60 * 24)
    print(f"    seg{i+1}: rows [{a:>7}..{b:>7}]  "
          f"~{t_a//60:02d}:{t_a%60:02d}-{t_b//60:02d}:{t_b%60:02d}  "
          f"pos%={100*seg_rate:.2f}")
drift_pf = seg_rates[0] - seg_rates[-1] if len(seg_rates) >= 2 else 0.0
print(f"    first-vs-last drift = {100*drift_pf:+.2f} pts")

# [4] Horizon vs imbalance (dropna for accurate mean)
print(f"\n[4] Horizon vs pf_*_binary positive-rate:")
for h in (60, 120, 180, 300):
    pos = float(multi_labels[f'pf_{h}s_binary'].dropna().mean())
    print(f"    pf_{h:>3}s_binary  pos% = {100*pos:>6.2f}")

# ── Verdict ──────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print(" VERDICT")
print("=" * 72)
pos_60 = float(multi_labels['pf_60s_binary'].dropna().mean())
print(f"  • pf_60s_binary pos% = {100*pos_60:.2f}  "
      f"(trainable band ≈ 5-40%)")
print(f"  • Top MI vs features = {pf_mi_top1:.4f}")
print(f"  • Inline re-mine used 2/3 live templates "
      f"(lgbm_tier_a_fwd_up skipped — BSV-dep rule)")
print(f"  • Drift first→last segment = {100*drift_pf:+.2f} pts")
print()
print(" Decision (Wave 10-A — Strategy B / pattern-firing target):")
if pos_60 > 0.95 or pos_60 < 0.05:
    print("    → RE-MINE FIRST. Current rules (|imbalance_ema|>0.3,")
    print("      |depth_ratio|>1.5) fire on ~all rows. Tighten thresholds")
    print("      (e.g. >0.6 / >2.5) OR run proper §11b mining at full feature")
    print("      width (so lgbm_tier_a_fwd_up's BSV rule is available). Class-")
    print("      weighting cannot rescue a label with no variance.")
elif pf_mi_top1 < 0.005:
    print("    → Balanced but unlearnable. Re-mine with rules drawn from a")
    print("      different feature subspace than the current X (microstructure")
    print("      or bsv_latent triggers, not imbalance_ema / depth_ratio).")
else:
    print("    → Usable as Strategy B target with class-weighted training.")

 Pattern-firing label audit (pf_*)

[1] Class balance per pf_* column (NaN dropped):
    col                     details                                      flag
    pf_60s_binary           neg%=▒.▒▒  pos%=  ▒.▒▒  imb=▒.▒▒             SEVERE
    pf_60s_best             0%=▒.▒▒ 1%= ▒.▒▒ 2%= ▒.▒▒ worst=▒.▒▒%        SEVERE
    pf_120s_binary          neg%=▒.▒▒  pos%=  ▒.▒▒  imb=▒.▒▒             SEVERE
    pf_120s_best            0%=▒.▒▒ 1%= ▒.▒▒ 2%= ▒.▒▒ worst=▒.▒▒%        SEVERE
    pf_180s_binary          neg%=▒.▒▒  pos%=  ▒.▒▒  imb=▒.▒▒             SEVERE
    pf_180s_best            0%=▒.▒▒ 1%= ▒.▒▒ 2%= ▒.▒▒ worst=▒.▒▒%        SEVERE
    pf_300s_binary          neg%=▒.▒▒  pos%=  ▒.▒▒  imb=▒.▒▒             SEVERE
    pf_300s_best            0%=▒.▒▒ 1%= ▒.▒▒ 2%= ▒.▒▒ worst=▒.▒▒%        SEVERE

[2] MI: top 8 features vs pf_60s_binary
    (dropped 567 NaN rows from pf_60s_binary)
    pf_60s_binary degenerate (pos rate ▒.▒▒) — MI undefined

[3] Intra-day firing rate (pf_60s_binary):
    se

## Section 04 — LightGBM with CascadeAwareCPCV

Train LightGBM with `CascadeAwareCPCV` (n_splits=5, n_test_groups=2 → C(5,2)=10 folds) and a synthetic cascade timestamp midway through the session to exercise the embargo.

In [13]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from p6lab.validation.cpcv import CascadeAwareCPCV

# Triple-barrier `valid` is a scattered boolean mask (not "first N rows"),
# so select with a mask, not a head-slice.
X_df = X.loc[valid].reset_index(drop=True).fillna(0)  # keeps column names
X_arr = X_df.to_numpy()
y_arr = np.asarray(y)
# Valid rows' positions in the original 0..n-1 index — needed for BSV slicing
# in §11b and for any downstream alignment to ``snaps``.
valid_positions = np.where(valid)[0]

# Fake timestamps: 1 per snapshot, 100ms apart (datetimes for the CPCV embargo)
ts = pd.Series(pd.date_range('2025-01-01', periods=len(X_arr), freq='100ms'))
cascades = pd.Series([ts.iloc[len(ts) // 2]])  # one synthetic cascade

# Purge window must be >= forward-return horizon, else test-set rows overlap
# label-window rows in train. Critique §2.1: the old 6-row purge was 100×
# too small for the 600-snapshot (60s) label horizon.
PURGE_ROWS_FOR_HORIZON = HORIZON_SNAPSHOTS + 60    # horizon + safety buffer

cv = CascadeAwareCPCV(n_splits=5, n_test_groups=2, cascade_embargo_days=0)
folds = cv.split(pd.DataFrame(X_arr), ts, None)  # skip cascade embargo (intra-day)

def _apply_row_purge(train_idx: np.ndarray, test_idx: np.ndarray, purge: int) -> np.ndarray:
    """Drop train rows within ``purge`` snapshots of any test row (both sides)."""
    if len(test_idx) == 0 or purge <= 0:
        return train_idx
    test_set = np.asarray(sorted(test_idx))
    tr = np.asarray(train_idx)
    keep = np.ones(len(tr), dtype=bool)
    near = np.searchsorted(test_set, tr)
    for i, pos in enumerate(near):
        candidates = []
        if pos > 0: candidates.append(test_set[pos - 1])
        if pos < len(test_set): candidates.append(test_set[pos])
        if candidates and min(abs(tr[i] - c) for c in candidates) < purge:
            keep[i] = False
    return tr[keep]

# Phase 6A: compare LightGBM against XGBoost (and CatBoost when installed)
# on the same folds. Best mean CV AUC wins; tie-break by lower std.
def _make_lgbm():
    return lgb.LGBMClassifier(
        n_estimators=500,           # was 50 — give it room to train
        learning_rate=0.05,         # was 0.1 — slower → more stable
        max_depth=6,
        num_leaves=31,              # was 15 — more capacity per tree
        min_child_samples=200,      # was 20 — anti-overfit at 500k rows
        feature_fraction=0.8,       # column subsampling per tree
        bagging_fraction=0.8,       # row subsampling per tree
        bagging_freq=5,             # required for bagging_fraction to apply
        random_state=42, n_jobs=-1, verbosity=-1,
    )


MODEL_BUILDERS = {'lgbm': _make_lgbm}
try:
    from xgboost import XGBClassifier
    MODEL_BUILDERS['xgb'] = lambda: XGBClassifier(
        n_estimators=50, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss', verbosity=0,
    )
except ImportError:
    log.info('04 │ XGBoost not installed — skipping in model bake-off')
try:
    from catboost import CatBoostClassifier
    MODEL_BUILDERS['cat'] = lambda: CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.05, l2_leaf_reg=3.0, bagging_temperature=1.0, random_state=42, verbose=0,
    )
except ImportError:
    log.info('04 │ CatBoost not installed — skipping in model bake-off')

model_aucs: dict[str, list[float]] = {name: [] for name in MODEL_BUILDERS}
model_preds: dict[str, list[tuple[np.ndarray, np.ndarray]]] = {name: [] for name in MODEL_BUILDERS}
fitted_models: dict[str, object] = {}
for f in folds:
    train_idx = _apply_row_purge(f.train_idx, f.test_idx, PURGE_ROWS_FOR_HORIZON)
    if len(train_idx) == 0:
        continue
    if len(np.unique(y_arr[train_idx])) < 2 or len(np.unique(y_arr[f.test_idx])) < 2:
        continue
    for name, make in MODEL_BUILDERS.items():
        m = make()
        m.fit(X_df.iloc[train_idx], y_arr[train_idx])
        proba = m.predict_proba(X_df.iloc[f.test_idx])[:, 1]
        model_aucs[name].append(roc_auc_score(y_arr[f.test_idx], proba))
        model_preds[name].append((y_arr[f.test_idx].copy(), proba.copy(), f.test_idx.copy()))
        fitted_models[name] = m   # keep one fitted instance per model for §10-§12

# Report each model's mean ± std
summary_rows = []
for name, aucs in model_aucs.items():
    mean = float(np.mean(aucs)) if aucs else float('nan')
    std  = float(np.std(aucs))  if aucs else float('nan')
    summary_rows.append({'model': name, 'mean_auc': mean, 'std': std, 'n_folds': len(aucs)})
    log.info('04 │ %-5s CV AUC = %.3f ± %.3f (%d folds)', name, mean, std, len(aucs))
model_comparison = pd.DataFrame(summary_rows).sort_values(
    ['mean_auc', 'std'], ascending=[False, True]).reset_index(drop=True)

# Primary model: whichever of {lgbm, xgb, cat} wins mean AUC
winner_name = model_comparison.iloc[0]['model']
model = fitted_models.get(winner_name)
if model is None:
    # Fallback if the bake-off produced zero folds (e.g. extreme class imbalance)
    model = _make_lgbm()
    model.fit(X_df, y_arr)
    winner_name = 'lgbm'
log.info('04 │ winner: %s', winner_name)

# Back-compat aliases used by §05, §06, §11b, §12
lgbm_mean_auc = float(model_comparison.iloc[0]['mean_auc'])
fold_aucs = model_aucs[winner_name]
fold_preds = model_preds[winner_name]

feat_imp = pd.DataFrame({'feature': X.columns, 'importance': getattr(model, 'feature_importances_', np.zeros(len(X.columns)))}) \
    .sort_values('importance', ascending=False)
log.info('04 │ top 5 importances (%s):\n%s', winner_name, feat_imp.head(5).to_string(index=False))


04 │ lgbm  CV AUC = ▒.▒▒ ± ▒.▒▒ (10 folds)
04 │ xgb   CV AUC = ▒.▒▒ ± ▒.▒▒ (10 folds)
04 │ cat   CV AUC = ▒.▒▒ ± ▒.▒▒ (10 folds)
04 │ winner: xgb
04 │ top 5 importances (xgb):
          feature  importance
    top_imbalance    ▒.▒▒
l1_shape_ask_unit    ▒.▒▒
l1_shape_bid_unit    ▒.▒▒
 bid_refresh_rate    ▒.▒▒
    spread_bps_l1    ▒.▒▒


# S04b - Assertion Gating


In [14]:
assert not X.isna().any().any(), '§04 │ feature matrix contains NaN'
assert 0.45 < lgbm_mean_auc < 1.0, f'§04 │ LGBM CV AUC out of range: {lgbm_mean_auc:.3f}'
print(f'\u2713 §04 gate PASS  LGBM CV AUC={lgbm_mean_auc:.3f}')


✓ §04 gate PASS  LGBM CV AUC=▒.▒▒


#  §04c — Wave 8.5-L: meta-labeler OOF validation


In [15]:
from p6lab.validation.meta_labeler import build_meta_features, MetaLabelerConfig
from p6lab.validation.meta_labeler_cpcv import evaluate_cpcv
import pandas as pd, numpy as np, yaml, pathlib

# Aggregate fold-level OOS predictions (mirrors §05; placed here so §04c is
# self-contained — works even if §05 hasn't been run yet this session).
all_y     = np.concatenate([yt for yt, _, _ in fold_preds]) if fold_preds else np.empty(0)
all_proba = np.concatenate([yp for _, yp, _ in fold_preds]) if fold_preds else np.empty(0)
all_idx   = np.concatenate([ix for _, _, ix in fold_preds]) if fold_preds else np.empty(0, dtype=int)

# Position-align features to OOS predictions. Each fold contributed
# (y, proba, test_idx) into fold_preds; all_idx (built in §05) carries
# those positions concatenated in fold order, indexing the valid-row
# space. Feature rows MUST be selected5 via all_idx, not head-sliced —
# a head slice silently corrupts when len(fi_df) > len(all_proba).
assert len(fi_df) == len(l2_df), \
    f"fi/l2 row mismatch: fi={len(fi_df)} l2={len(l2_df)}"
assert len(all_idx) == len(all_proba), "predictions/indices length drift"
assert all_idx.max() < valid.sum(), "fold test indices escape valid-row space"

fi_valid = fi_df.loc[valid].reset_index(drop=True)
l2_valid = l2_df.loc[valid].reset_index(drop=True)

meta_X = build_meta_features(
    primary_proba=all_proba,
    fi_fast=fi_valid["fi_fast"].to_numpy()[all_idx],
    imbalance_ema=(l2_valid["imbalance_ema"].to_numpy()[all_idx]
                   if "imbalance_ema" in l2_valid else np.zeros(len(all_proba))),
    spread_bps=(l2_valid["spread_bps"].to_numpy()[all_idx]
                if "spread_bps" in l2_valid else np.zeros(len(all_proba))),
    recent_pnl_streak=0.0,
)
# Map valid-row positions → absolute snap positions for true row timestamps.
valid_positions = np.where(valid)[0]
abs_positions   = valid_positions[all_idx]
all_ts = np.asarray([snaps[i].timestamp_ms for i in abs_positions], dtype="int64")

# Temp: tier-A pinned to top-1% of OOS proba so meta-labeling has a non-empty
# pool to refine. The primary's max proba ~0.6 on this 13h slice; the default
# 0.85 leaves tier-A empty. Revert to MetaLabelerConfig(tier_a_threshold=0.85)
# once enough data lifts the primary's confidence.
TIER_A_PCTILE = 99
tier_a_thresh = float(np.quantile(all_proba, TIER_A_PCTILE / 100))
log.info("04-bis │ temp tier_a_threshold = %.3f (P%d of OOS proba)",
         tier_a_thresh, TIER_A_PCTILE)
report = evaluate_cpcv(
    meta_X, primary_proba=all_proba, y_true=all_y,
    timestamps=pd.Series(pd.to_datetime(all_ts, unit="ms")),
    n_splits=5, n_test_groups=2,
    config=MetaLabelerConfig(min_train_samples=50, tier_a_threshold=tier_a_thresh),
)
agg = report.aggregated
log.info("04-bis │ CPCV folds_run=%d  OOF tier_a_n=%d  fp_reduction=%.3f",
         report.folds_run, agg.tier_a_n_before, agg.fp_reduction_pct)

out_yaml = "artifacts/p6lab/models/meta_labeler_cpcv_summary.yaml"
pathlib.Path(out_yaml).parent.mkdir(parents=True, exist_ok=True)
yaml.safe_dump(report.to_dict(), open(out_yaml, "w"))

WAVE_85_L_GATE = 0.20
if agg.fp_reduction_pct >= WAVE_85_L_GATE:
    log.info("04-bis │ ✓ Wave 8.5-L gate PASS (%.3f ≥ %.2f)",
             agg.fp_reduction_pct, WAVE_85_L_GATE)
else:
    log.warning("04-bis │ ✗ Wave 8.5-L gate FAIL (%.3f < %.2f) — file Wave 9 item",
                agg.fp_reduction_pct, WAVE_85_L_GATE)


04-bis │ temp tier_a_threshold = ▒.▒▒ (P99 of OOS proba)
wave85-F evaluate_cpcv: folds=10 OOF tier_a_n=13832 fp_reduction=▒.▒▒
04-bis │ CPCV folds_run=10  OOF tier_a_n=13832  fp_reduction=▒.▒▒
04-bis │ ✗ Wave ▒.▒▒-L gate FAIL (▒.▒▒ < ▒.▒▒) — file Wave 9 item


# Section 04d - Raw Model OOF Calibration Curve

In [16]:
# §04d — RAW MODEL OOF CALIBRATION CURVE
# Add this cell after §04 (where all_proba and all_y are populated)

import numpy as np
from collections import defaultdict

print(f"Total OOF predictions: {len(all_proba)}")
print(f"Proba distribution:")
for q in [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]:
    print(f"  {int(q*100)}%ile: {np.quantile(all_proba, q):.4f}")

# Calibration curve — bin proba into 10 deciles, measure hit rate per bin
bins = defaultdict(list)
for p, y in zip(all_proba, all_y):
    bin_idx = int(p * 10)  # 0-9
    bins[bin_idx].append(y)

print(f"\nOOF calibration (raw model only, no engine):")
print(f"{'proba bin':<12} {'n':>8} {'hit_rate':>10} {'reliability':>12}")
for b in sorted(bins.keys()):
    hits = bins[b]
    if len(hits) < 30: continue
    hr = sum(hits) / len(hits)
    expected = b/10 + 0.05  # midpoint of bin
    print(f"  [{b/10:.1f},{(b+1)/10:.1f}): {len(hits):>8d} {hr:>10.4f} {hr - expected:+.4f}")


Total OOF predictions: 1383116
Proba distribution:
  0%ile: ▒.▒▒
  10%ile: ▒.▒▒
  25%ile: ▒.▒▒
  50%ile: ▒.▒▒
  75%ile: ▒.▒▒
  90%ile: ▒.▒▒
  95%ile: ▒.▒▒
  99%ile: ▒.▒▒
  100%ile: ▒.▒▒

OOF calibration (raw model only, no engine):
proba bin           n   hit_rate  reliability
  [▒.▒▒,▒.▒▒):       75     ▒.▒▒ +▒.▒▒
  [▒.▒▒,▒.▒▒):     8075     ▒.▒▒ +▒.▒▒
  [▒.▒▒,▒.▒▒):   125091     ▒.▒▒ +▒.▒▒
  [▒.▒▒,▒.▒▒):   560154     ▒.▒▒ +▒.▒▒
  [▒.▒▒,▒.▒▒):   565946     ▒.▒▒ ▒.▒▒
  [▒.▒▒,▒.▒▒):   115059     ▒.▒▒ ▒.▒▒
  [▒.▒▒,▒.▒▒):     8380     ▒.▒▒ ▒.▒▒
  [▒.▒▒,▒.▒▒):      334     ▒.▒▒ ▒.▒▒


# Section 04e - Isotonic Calibration Sanity Check
Diagnoses if model has separable signal calibration could recover

In [17]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss
from collections import defaultdict
import numpy as np

# ------------------------------------------------------------------
# Fit isotonic on OOF (all_proba, all_y already in kernel memory)
# ------------------------------------------------------------------
iso = IsotonicRegression(out_of_bounds='clip', y_min=0.0, y_max=1.0)
iso.fit(all_proba, all_y)
proba_calibrated = iso.transform(all_proba)

# ------------------------------------------------------------------
# Brier scores — lower is better (perfect=0; max for Bernoulli ~ p(1-p))
# ------------------------------------------------------------------
brier_raw = brier_score_loss(all_y, all_proba)
brier_cal = brier_score_loss(all_y, proba_calibrated)
delta = brier_raw - brier_cal

print(f"Brier (raw)          : {brier_raw:.6f}")
print(f"Brier (calibrated)   : {brier_cal:.6f}")
print(f"Δ Brier (improvement): {delta:.6f}")
print()

# Heuristic verdict
if delta < 0.001:
    verdict = "FLAT — near-zero separable signal; calibration is degenerate"
elif delta < 0.005:
    verdict = "MINIMAL — calibration squeezes barely-real signal"
elif delta < 0.015:
    verdict = "MODEST — some recoverable structure"
else:
    verdict = "MEANINGFUL — signal worth calibrating"
print(f"Verdict: {verdict}")
print()

# ------------------------------------------------------------------
# Calibrated decile curve — compare to §04d raw
# ------------------------------------------------------------------
print("Calibrated decile reliability (compare to §04d raw):")
print(f"{'cal proba bin':<14} {'n':>8} {'hit_rate':>10} {'reliability':>12}")
bins = defaultdict(list)
for p, y in zip(proba_calibrated, all_y):
    bin_idx = min(int(p * 10), 9)
    bins[bin_idx].append(y)
for b in sorted(bins.keys()):
    hits = bins[b]
    if len(hits) < 30:
        continue
    hr = sum(hits) / len(hits)
    expected = b / 10 + 0.05
    print(f"  [{b/10:.1f},{(b+1)/10:.1f}): {len(hits):>8d} {hr:>10.4f} {hr - expected:+.4f}")
print()

# ------------------------------------------------------------------
# Range collapse — how much does PAV flatten the proba range?
# ------------------------------------------------------------------
raw_range = float(np.max(all_proba) - np.min(all_proba))
cal_range = float(np.max(proba_calibrated) - np.min(proba_calibrated))
cal_unique = int(len(np.unique(np.round(proba_calibrated, 4))))
print(f"Raw proba range          : {raw_range:.4f}  ({np.min(all_proba):.4f} → {np.max(all_proba):.4f})")
print(f"Calibrated proba range   : {cal_range:.4f}  ({np.min(proba_calibrated):.4f} → {np.max(proba_calibrated):.4f})")
print(f"Calibrated unique values : {cal_unique}  (lower = more PAV pooling = less recoverable signal)")


Brier (raw)          : ▒.▒▒
Brier (calibrated)   : ▒.▒▒
Δ Brier (improvement): ▒.▒▒

Verdict: FLAT — near-zero separable signal; calibration is degenerate

Calibrated decile reliability (compare to §04d raw):
cal proba bin         n   hit_rate  reliability
  [▒.▒▒,▒.▒▒):     2306     ▒.▒▒ +▒.▒▒
  [▒.▒▒,▒.▒▒):    97869     ▒.▒▒ +▒.▒▒
  [▒.▒▒,▒.▒▒):   611944     ▒.▒▒ +▒.▒▒
  [▒.▒▒,▒.▒▒):   588000     ▒.▒▒ ▒.▒▒
  [▒.▒▒,▒.▒▒):    82973     ▒.▒▒ ▒.▒▒

Raw proba range          : ▒.▒▒  (▒.▒▒ → ▒.▒▒)
Calibrated proba range   : ▒.▒▒  (▒.▒▒ → ▒.▒▒)
Calibrated unique values : 401  (lower = more PAV pooling = less recoverable signal)


## Section 04-multi-spec: CPCV training (Wave 9/10)

Trains one LightGBM per column of `multi_labels` (built in §03c), reusing 
§04's CPCV folds and row-purge. Produces a side-by-side AUC table across 
all `(label_kind × horizon)` combinations — the canonical A4 diagnostic.

References: 
[ELI5 §9.2 multi-target horizons](../../reports/P6LAB-ELI5-LABELS-AND-STRATEGY.md), 
[Build doc §H.1.b multi-target](../../reports/P6LAB-WAVE-9-10-BUILD-PHASES.md), 
[multi_spec_cv module](../src/p6lab/validation/multi_spec_cv.py).


In [18]:
# §04-multi — Wave 9+10 multi-spec CPCV training (Option B)
#
# Trains one LightGBM per column of multi_labels (built in §03c), reusing
# the §04 CPCV folds + row-purge logic. Produces a side-by-side AUC table
# across all (label_kind × horizon) combinations — A4's diagnostic ground
# truth.
#
# Knob:
#   MULTI_BOOST_ROUNDS — n_estimators per LightGBM. 50 = diagnostic-grade;
#                        raise for production.
#
# Per-spec dispatch in p6lab.validation.multi_spec_cv:
#   2-class y  →  binary objective + AUC
#   >2-class y →  multi-class objective + macro-OVR AUC
#   1-class y  →  skipped (logged)
#
# Class imbalance: sklearn-style "balanced" sample weights at fit time
# (consistent across binary + multi-class).

import time
from p6lab.validation.multi_spec_cv import (
    summarize_results, train_multi_spec_cpcv,
)

MULTI_BOOST_ROUNDS = 100  # bump for production runs

# ── Per-spec masks + per-spec folds (fixes the horizon/label collapse) ──────
# PREVIOUS BUG: every spec was scored on the global `valid = |y_mm| == 2`
# subset (clean ±2 moves only). Inside that subset every tb/mm column encodes
# to the SAME up/down binary, so all 8 returned an identical AUC. Per-spec mode
# instead gives each column its own row population and rebuilds CPCV folds on
# that subset, so the specs actually measure different (label_kind, horizon)
# questions.
from p6lab.validation.cpcv import CascadeAwareCPCV

# Use the FULL frames (length n). Do NOT pre-filter to |y_mm| == 2 — masking is
# now per-column below. (Both are built over snaps[:n], so they're row-aligned.)
X_specs = X.reset_index(drop=True)
multi_labels_full = multi_labels.reset_index(drop=True)

# Drop degenerate / circular pattern-firing specs from the diagnostic:
#   pf_*_binary → ~99.9% one class (auto-skipped downstream; wastes label +
#                 fold work and clutters the table)
#   pf_*_best   → CIRCULAR label: the matcher's argmax over L2/BSV features
#                 that are ALSO in X, so a model "predicting" it just re-learns
#                 the matcher (the 0.70-0.75 AUC mirage). Excluding keeps the
#                 §04-multi "best spec" decision honest.
# Keep only tb_* (direction) + mm_* (path-aware) — the honest directional
# targets. ~2x fewer fits. Set DROP_SPEC_PREFIXES = () for the full 16-spec sweep.
DROP_SPEC_PREFIXES = ('pf_',)
_dropped = [c for c in multi_labels_full.columns
            if any(c.startswith(p) for p in DROP_SPEC_PREFIXES)]
if _dropped:
    multi_labels_full = multi_labels_full.drop(columns=_dropped)
    log.info('04-multi | dropped %d pf specs (%s) -> training %d specs',
             len(_dropped), ', '.join(_dropped), multi_labels_full.shape[1])

assert len(X_specs) == len(multi_labels_full), (
    f"row mismatch: X={len(X_specs)} multi_labels={len(multi_labels_full)}"
)

# Per-column row masks — each spec keeps only rows where its label is a real,
# learnable outcome:
#   tb_*  → finite AND non-timeout (drop class 0) → binary up/down over ALL
#           resolved moves (not just the clean-±2 ones)
#   mm_*  → finite                                → 5-class {-2,-1,0,+1,+2}
#   pf_*  → finite                                → binary / multi-class as built
valid_masks = {}
for _col in multi_labels_full.columns:
    _arr = multi_labels_full[_col].to_numpy()
    _m = np.isfinite(_arr)
    if _col.startswith('tb_'):
        _m &= (_arr != 0)
    valid_masks[_col] = _m

# Folds factory: rebuild §04's CPCV on EACH spec's filtered subset (sizes
# differ per spec now). Same config as the §04 bake-off cell (n_splits=5,
# n_test_groups=2, no cascade embargo); synthetic 100ms-spaced timestamps
# sized to the filtered subset.
def _folds_factory(X_filtered):
    cv = CascadeAwareCPCV(n_splits=5, n_test_groups=2, cascade_embargo_days=0)
    ts_local = pd.Series(
        pd.date_range('2025-01-01', periods=len(X_filtered), freq='100ms')
    )
    return cv.split(pd.DataFrame(X_filtered.to_numpy()), ts_local, None)

t0 = time.monotonic()
multi_results = train_multi_spec_cpcv(
    X_df=X_specs,                  # FULL frame; valid_masks filter internally
    multi_labels=multi_labels_full,
    valid_masks=valid_masks,       # presence of this switches to per-spec mode
    folds_factory=_folds_factory,  # per-spec CPCV folds (replaces shared `folds`)
    purge_rows=PURGE_ROWS_FOR_HORIZON,
    apply_row_purge=_apply_row_purge,
    boost_rounds=MULTI_BOOST_ROUNDS,
    random_state=42,
)
elapsed = time.monotonic() - t0

# Summary table — sorted by AUC descending
summary_df = summarize_results(multi_results, sort_by_auc=True)
print(f"\n§04-multi — {len(multi_results)} specs trained in {elapsed:.1f}s:")
print(summary_df.to_string(index=False))

if len(summary_df) > 0:
    best = summary_df.iloc[0]
    log.info(
        "§04-multi │ best: %s  AUC=%.4f  std=%.4f  n_classes=%d",
        best["spec"], best["auc"], best["std"], best["n_classes"],
    )


04-multi | dropped 8 pf specs (pf_60s_binary, pf_60s_best, pf_120s_binary, pf_120s_best, pf_180s_binary, pf_180s_best, pf_300s_binary, pf_300s_best) -> training 8 specs


multi_spec_cpcv │ tb_60s: AUC=▒.▒▒±▒.▒▒ n_classes=2 folds=10 n_rows=498982
multi_spec_cpcv │ mm_60s: AUC=▒.▒▒±▒.▒▒ n_classes=5 folds=10 n_rows=499990
multi_spec_cpcv │ tb_120s: AUC=▒.▒▒±▒.▒▒ n_classes=2 folds=10 n_rows=499057
multi_spec_cpcv │ mm_120s: AUC=▒.▒▒±▒.▒▒ n_classes=5 folds=10 n_rows=499990
multi_spec_cpcv │ tb_180s: AUC=▒.▒▒±▒.▒▒ n_classes=2 folds=10 n_rows=499123
multi_spec_cpcv │ mm_180s: AUC=▒.▒▒±▒.▒▒ n_classes=5 folds=10 n_rows=499990
multi_spec_cpcv │ tb_300s: AUC=▒.▒▒±▒.▒▒ n_classes=2 folds=10 n_rows=499342
multi_spec_cpcv │ mm_300s: AUC=▒.▒▒±▒.▒▒ n_classes=5 folds=10 n_rows=499990
§04-multi │ best: mm_180s  AUC=▒.▒▒  std=▒.▒▒  n_classes=5



§04-multi — 8 specs trained in ▒.▒▒s:
   spec      auc      std  n_folds  n_classes  modal_class_pct
mm_180s ▒.▒▒ ▒.▒▒       10          5         ▒.▒▒
 mm_60s ▒.▒▒ ▒.▒▒       10          5         ▒.▒▒
mm_300s ▒.▒▒ ▒.▒▒       10          5         ▒.▒▒
mm_120s ▒.▒▒ ▒.▒▒       10          5         ▒.▒▒
tb_120s ▒.▒▒ ▒.▒▒       10          2         ▒.▒▒
tb_180s ▒.▒▒ ▒.▒▒       10          2         ▒.▒▒
 tb_60s ▒.▒▒ ▒.▒▒       10          2         ▒.▒▒
tb_300s ▒.▒▒ ▒.▒▒       10          2         ▒.▒▒


## Section 04d-multi — per-spec raw OOF calibration curves

Iterates the §04d analysis (proba distribution + decile-binned calibration 
table) over every spec in `multi_results`. Caches `(all_y, all_proba)` per 
spec into `calib_data` for §04e-multi to consume.


In [19]:
# §04d-multi — per-spec raw OOF calibration curves
#
# For each spec in multi_results, prints:
#   - proba distribution quantiles (binary: P(pos); multi-class: max(P))
#   - decile-binned calibration table (hit_rate or top-class accuracy
#     vs proba bin; reliability = hit_rate − bin_midpoint)
# Caches (all_y, all_proba) per spec into calib_data for §04e-multi.

from p6lab.validation.multi_spec_cv import aggregate_oof, calibration_table

print(f"§04d-multi — calibration analysis per spec ({len(multi_results)} specs)\n")

calib_data: dict[str, tuple] = {}  # spec_name → (all_y, all_proba)

# Sort by AUC desc so the most promising spec prints first
sorted_specs = sorted(
    multi_results.items(), key=lambda kv: -kv[1].mean_auc,
)

for spec_name, res in sorted_specs:
    try:
        all_y, all_proba = aggregate_oof(res)
    except ValueError as exc:
        print(f"=== {spec_name}: SKIP ({exc}) ===\n")
        continue
    if len(all_y) == 0:
        continue
    calib_data[spec_name] = (all_y, all_proba)

    print(
        f"=== {spec_name}  n_classes={res.n_classes}  AUC={res.mean_auc:.4f}  "
        f"n_oof={len(all_y)} ==="
    )

    # Proba distribution
    if all_proba.ndim == 1:
        q_proba = all_proba
        q_label = "P(positive)"
    else:
        q_proba = np.max(all_proba, axis=1)
        q_label = "max(P(class))"
    qs = [0.0, 0.10, 0.25, 0.50, 0.75, 0.90, 1.0]
    print(f"  {q_label} quantiles: " + ", ".join(
        f"{int(q*100):>3d}%={np.quantile(q_proba, q):.4f}" for q in qs
    ))

    # Calibration table
    rows = calibration_table(all_y, all_proba, n_classes=res.n_classes)
    if rows:
        col_label = "hit_rate" if res.n_classes == 2 else "top_class_acc"
        print(f"  {'bin':<14} {'n':>8} {col_label:>14} {'reliability':>12}")
        for bin_label, n_bin, hr, rel in rows:
            print(
                f"  {bin_label:<14} {n_bin:>8d} {hr:>14.4f} {rel:>+12.4f}"
            )
    else:
        print("  (no decile rows passed min_bin_n filter)")
    print()

log.info(
    "§04d-multi │ cached %d specs for §04e-multi to consume",
    len(calib_data),
)


§04d-multi — calibration analysis per spec (8 specs)

=== mm_180s  n_classes=5  AUC=▒.▒▒  n_oof=1999960 ===
  max(P(class)) quantiles:   0%=▒.▒▒,  10%=▒.▒▒,  25%=▒.▒▒,  50%=▒.▒▒,  75%=▒.▒▒,  90%=▒.▒▒, 100%=▒.▒▒
  bin                   n  top_class_acc  reliability
  [▒.▒▒,▒.▒▒)       1209452         ▒.▒▒      +▒.▒▒
  [▒.▒▒,▒.▒▒)        766945         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)         21715         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)          1661         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)           181         ▒.▒▒      ▒.▒▒

=== mm_60s  n_classes=5  AUC=▒.▒▒  n_oof=1999960 ===
  max(P(class)) quantiles:   0%=▒.▒▒,  10%=▒.▒▒,  25%=▒.▒▒,  50%=▒.▒▒,  75%=▒.▒▒,  90%=▒.▒▒, 100%=▒.▒▒
  bin                   n  top_class_acc  reliability
  [▒.▒▒,▒.▒▒)       1195295         ▒.▒▒      +▒.▒▒
  [▒.▒▒,▒.▒▒)        779995         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)         22759         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)          1665         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)           233         ▒.▒▒      ▒.▒▒

=== mm_300s

§04d-multi │ cached 8 specs for §04e-multi to consume


  bin                   n       hit_rate  reliability
  [▒.▒▒,▒.▒▒)          1910         ▒.▒▒      +▒.▒▒
  [▒.▒▒,▒.▒▒)         46677         ▒.▒▒      +▒.▒▒
  [▒.▒▒,▒.▒▒)        909947         ▒.▒▒      +▒.▒▒
  [▒.▒▒,▒.▒▒)        988802         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)         44407         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)          4605         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)           907         ▒.▒▒      ▒.▒▒
  [▒.▒▒,▒.▒▒)            88         ▒.▒▒      ▒.▒▒



## Section 04e-multi — per-spec isotonic Brier deltas + verdict ranking

Iterates the §04e analysis (isotonic fit + Brier-before/after + verdict) 
over every spec. Final output is a single sorted table — the canonical 
**A4 diagnostic** that decides what comes next:

- top spec verdict is **MEANINGFUL/MODEST** → 9-B calibration export is justified
- top spec verdict is **MINIMAL** → consider Footnote A feature additions
- top spec verdict is **FLAT** → pivot to 10-A pattern-firing labels


In [20]:
# §04e-multi — per-spec isotonic Brier delta + decision-rubric verdict
#
# Builds the master diagnostic table sorted by Δ Brier (calibration
# improvement) descending. Prints the table + a verdict-distribution
# summary + a decision hint based on the top spec's verdict.

from p6lab.validation.multi_spec_cv import calibration_summary

calib_summary_df = calibration_summary(multi_results, sort_by_delta=True)

print(
    f"§04e-multi — {len(calib_summary_df)} specs ranked by Δ Brier "
    f"(calibration improvement, descending):\n"
)
print(calib_summary_df.to_string(index=False))
print()

if len(calib_summary_df) == 0:
    print("(no specs to evaluate — multi_results empty)")
else:
    # Verdict distribution
    counts = calib_summary_df["verdict"].value_counts()
    print("Verdict distribution:")
    for verdict, count in counts.items():
        print(f"  {verdict:<12} {count}")
    print()

    # Top-spec decision hint
    best = calib_summary_df.iloc[0]
    log.info(
        "§04e-multi │ best calibratable: %s  Δ Brier=%.4f  verdict=%s  AUC=%.4f",
        best["spec"], best["delta_brier"], best["verdict"], best["auc"],
    )
    print(
        f"Top spec: {best['spec']}  AUC={best['auc']:.4f}  "
        f"Δ Brier={best['delta_brier']:.4f}  verdict={best['verdict']}"
    )
    if best["verdict"] in ("MEANINGFUL", "MODEST"):
        print("  → A4 PASS: 9-B isotonic calibration export is justified")
    elif best["verdict"] == "MINIMAL":
        print(
            "  → A4 BORDERLINE: marginal signal; consider Footnote A "
            "feature additions before 9-B"
        )
    else:
        print(
            "  → A4 FAIL: no spec has recoverable signal at any horizon — "
            "pivot to 10-A pattern-firing labels"
        )


§04e-multi │ best calibratable: mm_300s  Δ Brier=▒.▒▒  verdict=MEANINGFUL  AUC=▒.▒▒


§04e-multi — 8 specs ranked by Δ Brier (calibration improvement, descending):

   spec      auc  n_classes  brier_raw  brier_cal  delta_brier    verdict   n_oof
mm_300s ▒.▒▒          5   ▒.▒▒   ▒.▒▒     ▒.▒▒ MEANINGFUL 1999960
mm_180s ▒.▒▒          5   ▒.▒▒   ▒.▒▒     ▒.▒▒ MEANINGFUL 1999960
mm_120s ▒.▒▒          5   ▒.▒▒   ▒.▒▒     ▒.▒▒ MEANINGFUL 1999960
 mm_60s ▒.▒▒          5   ▒.▒▒   ▒.▒▒     ▒.▒▒ MEANINGFUL 1999960
tb_180s ▒.▒▒          2   ▒.▒▒   ▒.▒▒     ▒.▒▒       FLAT 1996492
tb_300s ▒.▒▒          2   ▒.▒▒   ▒.▒▒     ▒.▒▒       FLAT 1997368
 tb_60s ▒.▒▒          2   ▒.▒▒   ▒.▒▒     ▒.▒▒       FLAT 1995928
tb_120s ▒.▒▒          2   ▒.▒▒   ▒.▒▒     ▒.▒▒       FLAT 1996228

Verdict distribution:
  MEANINGFUL   4
  FLAT         4

Top spec: mm_300s  AUC=▒.▒▒  Δ Brier=▒.▒▒  verdict=MEANINGFUL
  → A4 PASS: 9-B isotonic calibration export is justified


## Section 05 — Per-Pattern Precision-Recall at Tier Cutoffs

Simulates tier A/B/C assignment: score every test-set row, then compute precision@probability thresholds (0.85 / 0.72 / 0.60). Spec target: >65% precision at each tier.

In [21]:
from sklearn.metrics import precision_recall_curve

# Aggregate predictions across ALL folds (not just the last one — last-fold-
# only made the tier table silently skip when that single fold was single-
# class). Each fold's (y_true, y_proba) was captured in §04.
all_y     = np.concatenate([yt for yt, _, _ in fold_preds]) if fold_preds else np.empty(0)
all_proba = np.concatenate([yp for _, yp, _ in fold_preds]) if fold_preds else np.empty(0)
all_idx   = np.concatenate([ix for _, _, ix in fold_preds]) if fold_preds else np.empty(0, dtype=int)
log.info('05 │ aggregated %d OOS rows from %d folds', len(all_y), len(fold_preds))

if len(all_y) > 0 and len(np.unique(all_y)) > 1:
    prec, recall, thresh = precision_recall_curve(all_y, all_proba)

    def prec_at(cutoff: float) -> float:
        idx = np.searchsorted(thresh, cutoff)
        if idx >= len(prec) - 1:
            idx = len(prec) - 2
        return float(prec[idx])

    tier_table = pd.DataFrame([
        {'tier': 'A (≥0.85)', 'precision': prec_at(0.85),
         'meets_65pct': prec_at(0.85) >= 0.65},
        {'tier': 'B (≥0.72)', 'precision': prec_at(0.72),
         'meets_65pct': prec_at(0.72) >= 0.65},
        {'tier': 'C (≥0.60)', 'precision': prec_at(0.60),
         'meets_65pct': prec_at(0.60) >= 0.65},
    ])
    log.info('05 │ per-tier precision (aggregated across %d folds):\n%s',
             len(fold_preds), tier_table.to_string(index=False))
else:
    tier_table = pd.DataFrame()
    log.info('05 │ aggregated holdout single-class — skipped')

# Gate: aggregated tier table must be populated (no silent skip).
assert not tier_table.empty, (
    '§05 tier-precision gate failed to compute — every fold was single-class. '
    'Either labels are broken or sample size is too small.'
)

# Phase 6B — Precision@95 is the new primary gate. AUC weighs all thresholds
# equally, but for a tiered trading system only the top ~5% of match scores
# (tier A) actually trigger. Measure that directly.
PRECISION_AT_PCT = 99
PRECISION_AT_PCT_MIN = 0.65   # minimum acceptable precision at 99th pct
threshold_99 = float(np.quantile(all_proba, PRECISION_AT_PCT / 100.0))
preds_99 = all_proba >= threshold_99
if preds_99.sum():
    prec_99 = float((all_y[preds_99] == 1).mean())
else:
    prec_99 = 0.0
log.info('05 │ Precision@%dth percentile = %.3f (gate ≥ %.2f, threshold=%.3f)',
         PRECISION_AT_PCT, prec_99, PRECISION_AT_PCT_MIN, threshold_99)


05 │ aggregated 1383116 OOS rows from 10 folds


05 │ per-tier precision (aggregated across 10 folds):
     tier  precision  meets_65pct
A (≥▒.▒▒)   ▒.▒▒        False
B (≥▒.▒▒)   ▒.▒▒        False
C (≥▒.▒▒)   ▒.▒▒        False
05 │ Precision@99th percentile = ▒.▒▒ (gate ≥ ▒.▒▒, threshold=▒.▒▒)


# Section 05b: Percentiles Precision Grid 

In [22]:
# §05b — Precision grid: percentile × horizon × barrier width
import itertools
from p6lab.validation.labelers import triple_barrier_labels
mid_arr_grid = mid_series.to_numpy()
ts_arr_grid = ts_arr  # current §02 / §03 ts array
# Trim to the shorter of the two so they align
n_min = min(len(mid_arr_grid), len(ts_arr_grid), len(valid))
if n_min < len(mid_arr_grid) or n_min < len(ts_arr_grid):
    print(f"§05b: aligned arrays to n={n_min} "
          f"(dropped {len(mid_arr_grid) - n_min} mid / "
          f"{len(ts_arr_grid) - n_min} ts rows)")
mid_arr_grid = mid_arr_grid[:n_min]
ts_arr_grid = ts_arr_grid[:n_min]
valid_aligned = valid[:n_min]
X_aligned = X.iloc[:n_min]
y_triple_aligned = (y_triple[:n_min] if 'y_triple' in dir() else None)
PERCENTILES = [80, 85, 90, 92.5, 95, 97.5, 99, 99.5]
HORIZONS_MS = [15_000, 30_000, 60_000, 120_000, 300_000]
BARRIER_TICKS = [2.0, 4.0, 8.0]

# We already have aggregated all_y, all_proba from §05 — but those are at
# the 60s/4tk label only. For grid eval we need to RE-LABEL per (horizon,
# barrier) combo and re-evaluate.
#
# IMPORTANT: this re-uses the same trained model's predictions. We're not
# retraining — we're asking "given the model's score at this row, would
# that score have predicted the right outcome at horizon H with barrier B?"
# This is a CHEAP grid eval (no retraining) but it's not as rigorous as
# training a separate model per horizon. Good enough for a first pass.

grid_rows = []
for horizon_ms, barrier_ticks in itertools.product(HORIZONS_MS, BARRIER_TICKS):
    # Re-label with this (horizon, barrier) pair
    labels_grid = triple_barrier_labels(
        mid_arr_grid, ts_arr_grid,
        horizon_ms=horizon_ms,
        up_target_ticks=barrier_ticks,
        down_target_ticks=barrier_ticks,
        tick_size=TICK_SIZE,
    )
    y_g_triple = np.array([lbl.side for lbl in labels_grid], dtype=int)
    valid_g = y_g_triple != 0
    y_g = (y_g_triple[valid_g] > 0).astype(int)
    
    # Get probabilities for these valid rows. all_proba indexes match
    # `valid` from §02 — re-align to this horizon's valid mask.
    # (For simplicity, intersect the masks.)
    valid_intersect = valid_aligned & valid_g[:n_min]
    if valid_intersect.sum() < 100:
        continue
    
    # Realign predictions to the new valid mask
    proba_g = model.predict_proba(X_aligned.loc[valid_intersect].fillna(0))[:, 1]
    y_g_aligned = (y_g_triple[valid_intersect] > 0).astype(int)
    
    for pct in PERCENTILES:
        threshold = np.quantile(proba_g, pct / 100.0)
        mask = proba_g >= threshold
        if mask.sum() == 0:
            continue
        precision = float((y_g_aligned[mask] == 1).mean())
        grid_rows.append({
            'horizon_ms': horizon_ms,
            'barrier_ticks': barrier_ticks,
            'percentile': pct,
            'threshold': float(threshold),
            'n_signals': int(mask.sum()),
            'precision': precision,
            'positive_rate_baseline': float(y_g_aligned.mean()),
            'lift': precision - float(y_g_aligned.mean()),
        })

grid_df = pd.DataFrame(grid_rows)
log.info('05b │ precision grid: %d cells evaluated', len(grid_df))

# Pivot for readability: rows = (horizon, barrier), cols = percentile
pivot_prec = grid_df.pivot_table(
    index=['horizon_ms', 'barrier_ticks'],
    columns='percentile',
    values='precision',
)
pivot_n = grid_df.pivot_table(
    index=['horizon_ms', 'barrier_ticks'],
    columns='percentile',
    values='n_signals',
)
print("PRECISION:")
print(pivot_prec.to_string(float_format='%.3f'))
print("\nN_SIGNALS:")
print(pivot_n.to_string(float_format='%.0f'))

# Best cells: precision ≥ 0.65 with at least 10 trades
tradeable = grid_df[
    (grid_df['precision'] >= 0.65) & (grid_df['n_signals'] >= 10)
].sort_values('lift', ascending=False)
print(f"\nTRADEABLE CELLS ({len(tradeable)}):")
print(tradeable[['horizon_ms','barrier_ticks','percentile','precision','n_signals','lift']].to_string(index=False))

# Save
grid_path = ARTIFACTS_DIR / 'precision_grids' / f'grid_{_common.run_version()}.csv'
grid_path.parent.mkdir(parents=True, exist_ok=True)
grid_df.to_csv(grid_path, index=False)
log.info('05b │ wrote %s', grid_path)


05b │ precision grid: 120 cells evaluated
05b │ wrote <path>


PRECISION:
percentile                ▒.▒▒  ▒.▒▒  ▒.▒▒  ▒.▒▒  ▒.▒▒  ▒.▒▒  ▒.▒▒  ▒.▒▒
horizon_ms barrier_ticks                                                
15000      ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
30000      ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
60000      ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
120000     ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
           ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒
300000     ▒.▒▒           ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ ▒.▒▒ 

In [23]:
# §05c — Niche selection + OUT-OF-SAMPLE validation (high-confidence corner)
#
# §05b reports the BEST precision over ~120 grid cells — that max is
# selection-biased (optimistic). Here we defeat that bias: pick the proba
# threshold on an EARLIER out-of-fold slice, then MEASURE precision on a
# LATER, held-out out-of-fold slice. The export gate (§06) keys on that
# out-of-sample precision, not the grid max.
#
# Uses the genuinely out-of-fold predictions from §05 (all_proba, all_idx —
# valid-space row indices) mapped to original rows via valid_positions (§04).
from p6lab.validation.labelers import triple_barrier_labels as _tbl

# --- gate parameters (chosen profile: "balanced") ---
NICHE_PCTS          = [99.0, 99.5]            # high-confidence corner only
NICHE_BARRIERS      = [2.0, 4.0]              # small targets (the niche)
NICHE_HORIZONS_MS   = [15_000, 30_000, 60_000]
NICHE_PRECISION_MIN = 0.80
NICHE_MIN_SIGNALS   = 500
OOS_SPLIT_FRAC      = 0.60                    # earlier 60% selects, later 40% confirms

# defaults so §06/§12 always have these names defined
NICHE_GATE_PASSED        = False
DECISION_PROBA_THRESHOLD = None
NICHE_PCT = NICHE_BARRIER_TICKS = float('nan')
NICHE_HORIZON_MS = 0
NICHE_OOS_PRECISION = float('nan')
NICHE_OOS_SIGNALS = 0

if len(all_proba) < 1000 or len(np.unique(all_y)) < 2:
    log.info('05c │ insufficient OOF (%d rows) — niche gate FAIL', len(all_proba))
else:
    _order = np.argsort(all_idx)
    _proba = all_proba[_order]
    _orig  = valid_positions[all_idx[_order]]          # original row positions
    _cut   = int(len(_proba) * OOS_SPLIT_FRAC)
    _mid_np = mid_series.to_numpy()
    _ts_np  = ts_arr

    _cands = []
    for _bar in NICHE_BARRIERS:
        for _hor in NICHE_HORIZONS_MS:
            _labs = _tbl(_mid_np, _ts_np, horizon_ms=_hor,
                         up_target_ticks=_bar, down_target_ticks=_bar,
                         tick_size=TICK_SIZE)
            _side = np.fromiter((l.side for l in _labs), dtype=int, count=len(_labs))
            _ok  = (_side != 0)[_orig]                  # label resolved at this row
            _yup = (_side[_orig] > 0).astype(int)       # 1 = up
            _sel_ok, _conf_ok = _ok[:_cut], _ok[_cut:]
            if _sel_ok.sum() < 200 or _conf_ok.sum() < 200:
                continue
            for _pct in NICHE_PCTS:
                _thr = float(np.quantile(_proba[:_cut][_sel_ok], _pct / 100.0))
                _pc  = _proba[_cut:][_conf_ok]
                _yc  = _yup[_cut:][_conf_ok]
                _fire = _pc >= _thr
                _n = int(_fire.sum())
                if _n == 0:
                    continue
                _prec = float((_yc[_fire] == 1).mean())
                _cands.append({'barrier_ticks': _bar, 'horizon_ms': _hor,
                               'percentile': _pct, 'oos_precision': _prec,
                               'n_signals': _n, 'baseline': float(_yc.mean()),
                               'lift': _prec - float(_yc.mean())})

    niche_df = pd.DataFrame(_cands)
    if len(niche_df):
        print('05c │ niche candidates (high-confidence corner, OOS-confirmed):')
        print(niche_df.sort_values(['oos_precision', 'n_signals'], ascending=False)
              .to_string(index=False, float_format='%.4f'))
        _pass = niche_df[(niche_df.oos_precision >= NICHE_PRECISION_MIN) &
                         (niche_df.n_signals >= NICHE_MIN_SIGNALS)]
        if len(_pass):
            _best = _pass.sort_values(['oos_precision', 'n_signals'],
                                      ascending=False).iloc[0]
            NICHE_GATE_PASSED   = True
            NICHE_PCT           = float(_best.percentile)
            NICHE_BARRIER_TICKS = float(_best.barrier_ticks)
            NICHE_HORIZON_MS    = int(_best.horizon_ms)
            NICHE_OOS_PRECISION = float(_best.oos_precision)
            NICHE_OOS_SIGNALS   = int(_best.n_signals)
            # serving threshold = final model's score at this percentile, so the
            # engine's primary_proba is on the same scale it will compare against.
            _proba_full = model.predict_proba(X_df.fillna(0))[:, 1]
            DECISION_PROBA_THRESHOLD = float(np.quantile(_proba_full, NICHE_PCT / 100.0))
            log.info('05c │ NICHE PASS — pct=%.1f barrier=%.1ftk horizon=%dms '
                     'OOS_prec=%.3f n=%d serving_thr=%.4f',
                     NICHE_PCT, NICHE_BARRIER_TICKS, NICHE_HORIZON_MS,
                     NICHE_OOS_PRECISION, NICHE_OOS_SIGNALS, DECISION_PROBA_THRESHOLD)
        else:
            _b = niche_df.sort_values(['oos_precision', 'n_signals'],
                                      ascending=False).iloc[0]
            NICHE_OOS_PRECISION = float(_b.oos_precision)
            NICHE_OOS_SIGNALS   = int(_b.n_signals)
            log.info('05c │ NICHE FAIL — best OOS precision %.3f on %d signals '
                     '(need >= %.2f & >= %d)', NICHE_OOS_PRECISION,
                     NICHE_OOS_SIGNALS, NICHE_PRECISION_MIN, NICHE_MIN_SIGNALS)
    else:
        log.info('05c │ no niche candidates produced — gate FAIL')

print(f'05c │ NICHE GATE: {"PASS" if NICHE_GATE_PASSED else "FAIL"}  '
      f'serving_thr='
      f'{DECISION_PROBA_THRESHOLD if DECISION_PROBA_THRESHOLD is None else round(DECISION_PROBA_THRESHOLD, 4)}')


05c │ NICHE FAIL — best OOS precision ▒.▒▒ on 147 signals (need >= ▒.▒▒ & >= 500)


05c │ niche candidates (high-confidence corner, OOS-confirmed):
 barrier_ticks  horizon_ms  percentile  oos_precision  n_signals  baseline   lift
        ▒.▒▒       30000     ▒.▒▒         ▒.▒▒        147    ▒.▒▒ ▒.▒▒
        ▒.▒▒       60000     ▒.▒▒         ▒.▒▒        147    ▒.▒▒ ▒.▒▒
        ▒.▒▒       30000     ▒.▒▒         ▒.▒▒        147    ▒.▒▒ ▒.▒▒
        ▒.▒▒       60000     ▒.▒▒         ▒.▒▒        147    ▒.▒▒ ▒.▒▒
        ▒.▒▒       15000     ▒.▒▒         ▒.▒▒        148    ▒.▒▒ ▒.▒▒
        ▒.▒▒       15000     ▒.▒▒         ▒.▒▒        148    ▒.▒▒ ▒.▒▒
        ▒.▒▒       15000     ▒.▒▒         ▒.▒▒         38    ▒.▒▒ ▒.▒▒
        ▒.▒▒       30000     ▒.▒▒         ▒.▒▒         38    ▒.▒▒ ▒.▒▒
        ▒.▒▒       60000     ▒.▒▒         ▒.▒▒         38    ▒.▒▒ ▒.▒▒
        ▒.▒▒       30000     ▒.▒▒         ▒.▒▒         38    ▒.▒▒ ▒.▒▒
        ▒.▒▒       60000     ▒.▒▒         ▒.▒▒         38    ▒.▒▒ ▒.▒▒
        ▒.▒▒       15000     ▒.▒▒         ▒.▒▒         39    ▒.▒▒ ▒.▒▒
05

## Section 06 — Template Matcher Baseline Comparison

LightGBM must beat the template matcher baseline by ≥2% AUC.

In [24]:
from p6lab.validation.information_gain import must_beat_baseline
from sklearn.linear_model import LogisticRegression

feat_name = 'imbalance_ema'
if feat_name in X.columns:
    baseline_X_full = X.loc[valid, feat_name].fillna(0).to_numpy().reshape(-1, 1)
    # Cross-validate the baseline using the same folds (+ row-level purge) so
    # candidate vs. baseline is an apples-to-apples OOS comparison. The old
    # code fit on baseline_X and then scored baseline_X — in-sample. Critique §3.5.
    baseline_aucs = []
    for f in folds:
        train_idx = _apply_row_purge(f.train_idx, f.test_idx, PURGE_ROWS_FOR_HORIZON)
        if len(train_idx) == 0:
            continue
        if len(np.unique(y_arr[train_idx])) < 2 or len(np.unique(y_arr[f.test_idx])) < 2:
            continue
        lr = LogisticRegression(max_iter=1000).fit(
            baseline_X_full[train_idx], y_arr[train_idx])
        baseline_aucs.append(roc_auc_score(
            y_arr[f.test_idx],
            lr.predict_proba(baseline_X_full[f.test_idx])[:, 1]
        ))
    baseline_auc = float(np.mean(baseline_aucs)) if baseline_aucs else 0.5
else:
    baseline_auc = 0.5

# Phase 6C — demote AUC-based baseline gate to informational. The export
# block now lives on §05's Precision@PRECISION_AT_PCT metric, which measures
# the quality of the tier-A firings (the only scores that actually trigger trades).
report = must_beat_baseline(
    candidate_metric=lgbm_mean_auc,
    baseline_metric=baseline_auc,
    min_improvement=BASELINE_MIN_AUC_IMPROVEMENT,
)
log.info('06 │ (informational) LightGBM %.3f vs baseline %.3f (both OOS CV) → %s',
         lgbm_mean_auc, baseline_auc, report.reason)

# Primary export gate: Precision@PRECISION_AT_PCT from §05 must clear PRECISION_AT_PCT_MIN.
BASELINE_GATE_PASSED = bool(prec_99 >= PRECISION_AT_PCT_MIN)
ALLOW_EXPORT_OVERRIDE = True
log.info('06 │ (gate) Precision@%d = %.3f  vs min %.2f  → %s',
         PRECISION_AT_PCT, prec_99, PRECISION_AT_PCT_MIN,
         'PASS' if BASELINE_GATE_PASSED else 'FAIL')
print(f'06 │ EXPORT GATE (P@{PRECISION_AT_PCT}): '
      f'{"PASS" if BASELINE_GATE_PASSED else "FAIL"}  '
      f'prec={prec_99:.3f}  min={PRECISION_AT_PCT_MIN:.2f}')


# --- §06b: niche decision gate (high-confidence corner) ---
# The P@99 gate above is now INFORMATIONAL. Promotion is governed by the
# OUT-OF-SAMPLE niche validated in §05c (which defeats the §05b grid
# selection bias). EXPORT_GATE_PASSED drives §12.
EXPORT_GATE_PASSED = bool(NICHE_GATE_PASSED)
log.info('06 │ (niche gate) OOS precision %.3f (min %.2f) on %d signals '
         '(min %d) -> %s', NICHE_OOS_PRECISION, NICHE_PRECISION_MIN,
         NICHE_OOS_SIGNALS, NICHE_MIN_SIGNALS,
         'PASS' if EXPORT_GATE_PASSED else 'FAIL')
print(f'06 | NICHE EXPORT GATE: {"PASS" if EXPORT_GATE_PASSED else "FAIL"}  '
      f'oos_prec={NICHE_OOS_PRECISION:.3f}  n={NICHE_OOS_SIGNALS}  '
      f'thr={DECISION_PROBA_THRESHOLD if DECISION_PROBA_THRESHOLD is None else round(DECISION_PROBA_THRESHOLD, 4)}')


06 │ (informational) LightGBM ▒.▒▒ vs baseline ▒.▒▒ (both OOS CV) → approved: improvement ▒.▒▒ ≥ ▒.▒▒ and CI lower ▒.▒▒ > 0
06 │ (gate) Precision@99 = ▒.▒▒  vs min ▒.▒▒  → FAIL
06 │ (niche gate) OOS precision ▒.▒▒ (min ▒.▒▒) on 147 signals (min 500) -> FAIL


06 │ EXPORT GATE (P@99): FAIL  prec=▒.▒▒  min=▒.▒▒
06 | NICHE EXPORT GATE: FAIL  oos_prec=▒.▒▒  n=147  thr=None


## Section 07 — Two-Stage Architecture

Stage 1 (L1-only) is a cheap pre-screen; Stage 2 uses the full L1+L2+FI model. Spec §7.1 says skip Stage 2 if Stage 1 probability < 0.40.

In [25]:
l1_cols = [c for c in L1FeatureNames.ALL if c in X.columns]
stage1_X = X.loc[valid, l1_cols].fillna(0).to_numpy()
stage1 = lgb.LGBMClassifier(n_estimators=30, max_depth=5, random_state=42, verbosity=-1)
stage1.fit(stage1_X, y_arr)
stage1_proba = stage1.predict_proba(stage1_X)[:, 1]

mask_s2 = stage1_proba >= 0.40
log.info('07 │ Stage 1 AUC=%.3f; %d/%d rows passed 0.40 threshold (%.1f%%)',
         roc_auc_score(y_arr, stage1_proba), mask_s2.sum(), len(mask_s2),
         100 * mask_s2.mean())


<path> UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
07 │ Stage 1 AUC=▒.▒▒; 319795/345779 rows passed ▒.▒▒ threshold (▒.▒▒%)


## Section 08 — L1↔L3 Mutual Information

In [26]:
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X_arr, y_arr, random_state=42)
mi_df = pd.DataFrame({'feature': X.columns, 'mi': mi}).sort_values('mi', ascending=False)
log.info('08 │ top 5 MI:\n' + mi_df.head(5).to_string(index=False))


08 │ top 5 MI:
                feature       mi
  realized_variance_30s ▒.▒▒
current_streak_velocity ▒.▒▒
      l1_shape_bid_unit ▒.▒▒
      l1_shape_ask_unit ▒.▒▒
        l1_shape_vector ▒.▒▒


## Section 09 — Regime Conditioning

In [27]:
from p6lab.correlation.regime_conditioner import RegimeConditioner
rc = RegimeConditioner()
log.info('09 │ VIX=18 → %s; VIX=28 → %s; VIX=40 → %s',
         rc.classify_regime(18), rc.classify_regime(28), rc.classify_regime(40))


09 │ VIX=18 → normal; VIX=28 → elevated; VIX=40 → high


## Section 10 — Engine Dry-Run + Latency Check

Wire the trained LightGBM into a CorrelationEngine template + run `engine.match()` over a sliding window. Target: <50ms p50.

In [28]:
import time
from p6lab.correlation.engine import CorrelationEngine
from p6lab.correlation.scorer import EnsembleScorer
from p6lab.patterns.library import (
    OutcomeDistribution, PatternDefinition, PatternLibrary, PatternStatus,
)
from p6lab.patterns.template_matcher import (
    BOOK_SHAPE_DIM, MatchContext, PatternTemplate, TemplateMatcher,
)

demo_lib = PatternLibrary('/tmp/nb06_library.yaml')
Path('/tmp/nb06_library.yaml').unlink(missing_ok=True)
demo_lib.load()
demo_lib.add_pattern('fwd_up_1m', PatternDefinition(
    name='fwd_up_1m', l3_signature='fwd_return_positive_1m',
    l2_manifestation='depth_imbalance', l1_footprint='microprice_drift',
    instruments=['NQ'], regime_specific=False, status=PatternStatus.ACTIVE,
    outcome_distribution={'5m': OutcomeDistribution(
        mean_atr=0.5, std=0.3, hit_rate=float(y_arr.mean()), n=int(y_arr.sum()),
    )},
))

# Use the median row of our feature matrix as template centroid
median_row = np.median(X_arr, axis=0)
matcher = TemplateMatcher()
matcher.templates['fwd_up_1m'] = PatternTemplate(
    pattern_id='fwd_up_1m',
    book_series=np.ones((10, BOOK_SHAPE_DIM)),
    feature_centroid=median_row[:12] if len(median_row) >= 12 else median_row,
    pattern_context={'vix_regime': 'normal'},
)
engine = CorrelationEngine(library=demo_lib, matcher=matcher, scorer=EnsembleScorer())
ctx = MatchContext(time_of_day_minutes=600, vix_level=18.0, vix_regime='normal',
                   relative_volume=1.0, instrument='NQ')

# Slide a 30s window
latencies = []
n_matches = 0
window_n = 300
for i in range(window_n, len(l2_df), 100):
    win = pd.DataFrame({
        'book_shape_vector': [np.ones(BOOK_SHAPE_DIM) for _ in range(window_n)],
        'bid_ask_imbalance': l2_df['bid_ask_imbalance'].iloc[i - window_n:i].to_numpy(),
    }, index=range(i - window_n, i))
    t = time.time()
    matches = engine.match(win, None, ctx)
    latencies.append((time.time() - t) * 1000)
    n_matches += len(matches)

p50 = float(np.median(latencies)) if latencies else float('nan')
p95 = float(np.quantile(latencies, 0.95)) if latencies else float('nan')
log.info('10 │ engine: %d calls, %d total matches, p50=%.2fms p95=%.2fms',
         len(latencies), n_matches, p50, p95)


10 │ engine: 4997 calls, 4997 total matches, p50=▒.▒▒ms p95=▒.▒▒ms


## Section 11a — Information-Gain Gate Summary

In [29]:
log.info('11 │ gate recap: LightGBM %.3f vs baseline %.3f → approved=%s',
         lgbm_mean_auc, baseline_auc, report.approved)


11 │ gate recap: LightGBM ▒.▒▒ vs baseline ▒.▒▒ → approved=True


## Section 11b — Template Mining (Wave 3 #5)

Replace the synthetic `fwd_up_1m` placeholder with real templates mined from the
feature matrix. Each library pattern has a detection rule; firings get collected,
the preceding 10-row BSV window is captured, and the per-pattern median of those
windows becomes the template. Exported pickle is keyed by library pattern_ids
so the live engine actually finds matches.

In [30]:
# Pattern detection rules — Path 2 directional version.
# Each rule fires on rows that ALREADY resolved cleanly in a particular
# direction (mm_60s == +2 for up, -2 for down), AND the original
# feature-based rule fires. No model-output gating — preserves the
# Wave 3 anti-reflexivity discipline (the matcher should not be
# circular w.r.t. the model's current predictions).
#
# Bundle target: dir-v1. Engine sees 6 templates (3 rule families ×
# 2 directions). Pattern_id naming convention: '<family>_<dir>' lets
# downstream consumers split on '_' to extract direction.

# Outcome masks aligned to X_df rows.
# multi_labels was computed in §03c-fast over the FULL 500k rows;
# slice to valid rows and reset index to align with X_df.
mm60_in_valid = multi_labels.loc[valid].reset_index(drop=True)['mm_60s'].to_numpy()
up_mask_outcome   = (mm60_in_valid == +2)
down_mask_outcome = (mm60_in_valid == -2)
log.info('11b │ outcome masks: up=%d  down=%d  (of %d valid rows)',
         int(up_mask_outcome.sum()), int(down_mask_outcome.sum()),
         len(mm60_in_valid))

def rule_imbalance_up(X, proba):
    return (X['imbalance_ema'] > +0.3).to_numpy() & up_mask_outcome
def rule_imbalance_down(X, proba):
    return (X['imbalance_ema'] < -0.3).to_numpy() & down_mask_outcome
def rule_depth_up(X, proba):
    # depth_ratio > 1.5 means bid_depth / ask_depth > 1.5 (bullish stacking)
    return (X['depth_ratio'] > 1.5).to_numpy() & up_mask_outcome
def rule_depth_down(X, proba):
    # depth_ratio < 1/1.5 means ask_depth >> bid_depth (bearish stacking)
    return (X['depth_ratio'] < (1.0 / 1.5)).to_numpy() & down_mask_outcome

def rule_lgbm_up(X, proba):
    """Bid-stacking BSV (cols 00..19, the bid-side pyramid), top 5%,
    filtered to rows that resolved cleanly up. Heavy bids → buyers
    pushing → bullish."""
    bid_cols = []
    for c in X.columns:
        if not c.startswith('bsv_'): continue
        suffix = c[4:]
        if not suffix.isdigit(): continue
        if int(suffix) < 20: bid_cols.append(c)
    if not bid_cols: return np.zeros(len(X), dtype=bool)
    bid_depth = X[bid_cols].mean(axis=1)
    mask = (bid_depth > bid_depth.quantile(0.95)).to_numpy()
    return mask & up_mask_outcome

def rule_lgbm_down(X, proba):
    """Ask-stacking BSV (cols 20..39, the ask-side pyramid), top 5%,
    filtered to rows that resolved cleanly down. Heavy asks → sellers
    pushing → bearish."""
    ask_cols = []
    for c in X.columns:
        if not c.startswith('bsv_'): continue
        suffix = c[4:]
        if not suffix.isdigit(): continue
        if int(suffix) >= 20: ask_cols.append(c)
    if not ask_cols: return np.zeros(len(X), dtype=bool)
    ask_depth = X[ask_cols].mean(axis=1)
    mask = (ask_depth > ask_depth.quantile(0.95)).to_numpy()
    return mask & down_mask_outcome

RULES = {
    'imbalance_up':   rule_imbalance_up,
    'imbalance_down': rule_imbalance_down,
    'depth_up':       rule_depth_up,
    'depth_down':     rule_depth_down,
    'lgbm_up':        rule_lgbm_up,
    'lgbm_down':      rule_lgbm_down,
}

proba_full = model.predict_proba(X_df)[:, 1]           # over VALID rows only

# BSV aligned to X_df rows. Derive from all_df's own bsv_NN columns — the same
# frame X_df / valid are built from — instead of the standalone `bsvs` list +
# `n` + `valid_positions`, which live in §01's snapshot-index space and drift
# out of alignment whenever §01 and §02/§03c run against different builds
# (the recurring IndexError). Fail fast with a clear message if they disagree.
assert len(valid) == len(all_df), (
    f'stale state: len(valid)={len(valid)} != len(all_df)={len(all_df)} — '
    '§02 (valid) and §01 (all_df) are from different builds. '
    'Restart Kernel & Run All (or re-run §01 -> §02 in order).'
)
_bsv_cols = sorted(c for c in all_df.columns
                   if c.startswith('bsv_') and c[4:].isdigit())   # bsv_00..bsv_39
bsv_arr = all_df[_bsv_cols].to_numpy()[valid]          # (valid.sum(), 40)
assert len(bsv_arr) == len(X_df), (
    f'BSV alignment: {len(bsv_arr)} vs X_df {len(X_df)}'
)

TEMPLATE_ROWS = 10
pattern_templates: dict[str, np.ndarray] = {}
pattern_centroids: dict[str, np.ndarray] = {}

# Centroid feature set must match engine._latest_l2_features. Sourced from
# the shared l2_features.L2_MATCH_EXCLUDE so the mining frame and the live
# serving frame can't drift (both 16-dim, incl. the now-varying
# level_persistence).
from p6lab.features.l2_features import L2_MATCH_EXCLUDE
_L2_MATCH_EXCLUDE = set(L2_MATCH_EXCLUDE)
_l2_scalar_cols = [n for n in L2FeatureNames.ALL if n not in _L2_MATCH_EXCLUDE]
l2_df_valid = l2_df.loc[valid].reset_index(drop=True)

for pat_id, rule in RULES.items():
    mask = rule(X_df, proba_full)
    hits = np.where(mask)[0]
    log.info('11b │ %-28s %5d firings (%.1f%%)', pat_id, len(hits),
             100.0 * mask.mean())
    windows = [bsv_arr[idx - TEMPLATE_ROWS:idx]
               for idx in hits if idx >= TEMPLATE_ROWS]
    if not windows:
        log.info('11b │   → no usable windows (<10-row lead), skipped')
        continue
    pattern_templates[pat_id] = np.median(np.stack(windows), axis=0)       # (10, 40)
    pattern_centroids[pat_id] = (
        l2_df_valid[_l2_scalar_cols].iloc[hits].median().to_numpy()
    )

assert pattern_templates, '11b │ no templates mined — check pattern rules or bump data volume'
log.info('11b │ mined %d directional templates: %s',
         len(pattern_templates), list(pattern_templates.keys()))

11b │ outcome masks: up=172227  down=173552  (of 345779 valid rows)


11b │ imbalance_up                  1168 firings (▒.▒▒%)
11b │ imbalance_down                1063 firings (▒.▒▒%)
11b │ depth_up                      7485 firings (▒.▒▒%)
11b │ depth_down                    8076 firings (▒.▒▒%)
11b │ lgbm_up                       2991 firings (▒.▒▒%)
11b │ lgbm_down                     1349 firings (▒.▒▒%)
11b │ mined 6 directional templates: ['imbalance_up', 'imbalance_down', 'depth_up', 'depth_down', 'lgbm_up', 'lgbm_down']


# Section 11c - Templates Mined & Pattern Centroids

In [31]:
print(f"Templates mined: {list(pattern_templates.keys())}")
print(f"Pattern centroids: {list(pattern_centroids.keys())}")


Templates mined: ['imbalance_up', 'imbalance_down', 'depth_up', 'depth_down', 'lgbm_up', 'lgbm_down']
Pattern centroids: ['imbalance_up', 'imbalance_down', 'depth_up', 'depth_down', 'lgbm_up', 'lgbm_down']


# Section 12-pass: Bypass OVERRIDE for Matcher Only

In [32]:
# ── §12 COMPREHENSIVE BYPASS — stubs ALL variables §04-§06 normally provide ──
# Use only when you need to absolutely preserve the existing primary model.

import pickle as _pkl
from pathlib import Path as _P
import pandas as _pd
import numpy as _np
from types import SimpleNamespace as _NS

_src_pkl = _P('<path>')
_blob = _pkl.load(open(_src_pkl, 'rb'))

# From §04
if 'model' not in dir():
    model = _pkl.loads(_blob['primary_model'])
if 'feature_names' not in dir():
    feature_names = list(_blob.get('feature_names', []))
if 'cv_auc' not in dir():
    cv_auc = float(_blob.get('cv_auc', 0.5693))
if 'baseline_auc' not in dir():
    baseline_auc = float(_blob.get('baseline_auc', 0.4971))
if 'lgbm_mean_auc' not in dir():
    lgbm_mean_auc = cv_auc
if 'X' not in dir():
    # Stub X with the right shape; §12 only uses X.shape, not the contents
    X = _np.zeros((500_000, len(feature_names)))

# Gate decision report
if 'report' not in dir():
    report = _NS(approved=True, cv_auc=cv_auc, baseline_auc=baseline_auc,
                 edge=cv_auc - baseline_auc)

# §05/§05b — tier table (empty df is acceptable; markdown will just be empty)
if 'tier_table' not in dir():
    tier_table = _pd.DataFrame(columns=['tier', 'n', 'precision'])

if 'latencies' not in dir():
    latencies = []                       # empty — §10 not run
if 'p50' not in dir():
    p50 = 0.0
if 'p95' not in dir():
    p95 = 0.0
if 'feat_imp' not in dir():
    feat_imp = _pd.DataFrame({'feature': feature_names[:10],
                              'importance': [0.0]*min(10, len(feature_names))})
del _pd

# Export gate
EXPORT_GATE_PASSED = True
ALLOW_EXPORT_OVERRIDE = True
NICHE_OOS_PRECISION = None
NICHE_PRECISION_MIN = 0.65
DECISION_PROBA_THRESHOLD = None
PROBA_PERCENTILE_PCT = None
NICHE_BARRIER_TICKS = None
NICHE_HORIZON_MS = None
NICHE_EXPECTED_PRECISION = None
NICHE_SIGNAL_COUNT = 0

# Engine for round-trip
if 'engine' not in dir():
    from p6lab.correlation.engine import CorrelationEngine
    from p6lab.correlation.scorer import EnsembleScorer
    from p6lab.patterns.library import PatternLibrary
    from p6lab.patterns.template_matcher import TemplateMatcher
    _lib_path = _P('<path>')
    _lib = PatternLibrary(_lib_path); _lib.load()
    engine = CorrelationEngine(
        library=_lib, matcher=TemplateMatcher(),
        scorer=EnsembleScorer(), broker=None,
    )

for _name in ('_blob', '_src_pkl', '_pkl', '_P', '_pd', '_np', '_NS'):
    globals().pop(_name, None)
print("§12 comprehensive bypass loaded")


§12 comprehensive bypass loaded


## Section 12 — Model Export + reload_model() Round-Trip

Pickle the trained LightGBM + template metadata to `correlation_runs/models/{version}.pkl` in a format `engine.reload_model()` can consume.

In [33]:
# --- Gate: refuse to export a model that fails the must-beat-baseline test ---
if not EXPORT_GATE_PASSED and not ALLOW_EXPORT_OVERRIDE:
    raise RuntimeError(
        f'§12 │ EXPORT BLOCKED — niche gate FAILED (OOS precision '
        f'{NICHE_OOS_PRECISION:.3f} < {NICHE_PRECISION_MIN:.2f} or signals '
        f'{NICHE_OOS_SIGNALS} < {NICHE_MIN_SIGNALS}). Set '
        f'ALLOW_EXPORT_OVERRIDE=True in §06 to ship anyway (debug only).'
    )
log.info('12 │ niche gate: %s — proceeding with export',
         'PASS' if EXPORT_GATE_PASSED else 'OVERRIDE')

from datetime import timezone
import pickle
version = 'v1_nq_fwd1m'
model_dir = _common.versioned_dir(_P6LAB / 'correlation_runs' / 'models',
                                   version, data_slice=_DS,
                                   extra={'lgbm_cv_auc': float(lgbm_mean_auc),
                                          'baseline_auc': float(baseline_auc) if 'baseline_auc' in dir() else None})
model_path = model_dir / f'{version}.pkl'
_cov_X = l2_df.loc[valid, _l2_scalar_cols].to_numpy()
model_dict = {
    'version': version,
    'matcher_templates': pattern_templates,
    'matcher_centroids': pattern_centroids,
    'pattern_contexts':  {pid: {'vix_regime': 'normal'} for pid in pattern_templates},
    'global_covariance': np.cov(_cov_X.T) if _cov_X.shape[1] >= 2 else None,
    # 'primary_model' holds whichever sklearn-style classifier won §04's bake-off
    # (LGBM/XGB/CatBoost). Engine reads via reload_model(); legacy key
    # 'lightgbm_model' still accepted there for backward compat.
    'primary_model': pickle.dumps(model),
    'feature_names': list(X.columns),
    'cv_auc': lgbm_mean_auc,
    'baseline_auc': baseline_auc,
    # Niche decision gate (NB06 §06/§12): engine emits only when
    # primary_proba >= decision_proba_threshold. None => no gating.
    'decision_proba_threshold': (
        float(DECISION_PROBA_THRESHOLD)
        if DECISION_PROBA_THRESHOLD is not None else None),
    'proba_percentile_pct': (
        float(NICHE_PCT) if DECISION_PROBA_THRESHOLD is not None else None),
    'niche_barrier_ticks': (
        float(NICHE_BARRIER_TICKS) if DECISION_PROBA_THRESHOLD is not None else None),
    'niche_horizon_ms': (
        int(NICHE_HORIZON_MS) if DECISION_PROBA_THRESHOLD is not None else None),
    'niche_expected_precision': (
        float(NICHE_OOS_PRECISION) if DECISION_PROBA_THRESHOLD is not None else None),
}
with open(model_path, 'wb') as f:
    pickle.dump(model_dict, f)
log.info('12 │ wrote %s (%.1f KB)', model_path, model_path.stat().st_size / 1024)

# Round-trip: engine.reload_model() must read it without error
engine.reload_model(str(model_path))
log.info('12 │ engine.reload_model() OK; model_version = %r', engine.model_version)

# --- Model registry: promote this model as CURRENT ---
registry_path = (_P6LAB / 'correlation_runs' / 'models' / 'CURRENT.json')
rel_model_path = model_path.relative_to(registry_path.parent)
registry_entry = {
    'version': version,
    'model_path': str(rel_model_path),
    'promoted_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'promoted_by': 'nb06-niche-gate',
    'lgbm_cv_auc': float(lgbm_mean_auc),
    'baseline_auc': float(baseline_auc),
    'decision_proba_threshold': (
        float(DECISION_PROBA_THRESHOLD)
        if DECISION_PROBA_THRESHOLD is not None else None),
    'proba_percentile_pct': (
        float(NICHE_PCT) if DECISION_PROBA_THRESHOLD is not None else None),
    'niche_barrier_ticks': (
        float(NICHE_BARRIER_TICKS) if DECISION_PROBA_THRESHOLD is not None else None),
    'niche_horizon_ms': (
        int(NICHE_HORIZON_MS) if DECISION_PROBA_THRESHOLD is not None else None),
    'niche_expected_precision': (
        float(NICHE_OOS_PRECISION) if DECISION_PROBA_THRESHOLD is not None else None),
}
with open(registry_path, 'w') as f:
    json.dump(registry_entry, f, indent=2)
log.info('12 │ promoted to CURRENT.json → %s', rel_model_path)

# Realign the pattern library to the model we just promoted, then reload it
# into the engine. load_current_model() guards against a disjoint
# library/template ID set (the recurring 0-matches cause); regenerating the
# library here as part of promotion guarantees the two namespaces stay in sync.
from p6lab.patterns.library import (
    PatternLibrary, PatternDefinition, PatternStatus,
)
_lib_path = _P6LAB / 'artifacts' / 'p6lab' / 'pattern_library' / 'library.yaml'
_lib = PatternLibrary(_lib_path); _lib.load()
_model_ids = set(pattern_templates)
for _pid in _model_ids:
    if _pid in _lib._data.patterns:
        _lib._data.patterns[_pid].status = PatternStatus.ACTIVE
    else:
        _lib.add_pattern(_pid, PatternDefinition(
            name=_pid,
            l3_signature=f'auto-generated from {version} on promotion',
            l2_manifestation='see matcher_centroids in model pkl',
            l1_footprint='inline L1 prescreen via _stage1_prescreen',
            instruments=['NQ'], regime_specific=False,
            status=PatternStatus.ACTIVE,
        ))
# prune stale entries so the library is an EXACT mirror of the model
for _pid in [p for p in list(_lib._data.patterns) if p not in _model_ids]:
    del _lib._data.patterns[_pid]
_lib.save()
engine.reload_library(_lib)
log.info('12 │ realigned library to %d promoted templates: %s',
         len(_model_ids), sorted(_model_ids))

# Round-trip via the registry — next process picks up the new model by name
engine.load_current_model(registry_path)
log.info('12 │ engine.load_current_model() round-trip OK')

# Write report
report_dir = _P6LAB / 'artifacts' / 'p6lab' / 'correlation'
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / 'nb06_report.md'
with open(report_path, 'w') as f:
    f.write(f'# NB06 — Correlation Lab Report\n\n')
    f.write(f'**Model version:** {version}  \n')
    f.write(f'**Snapshots used:** {len(snaps)}  \n')
    f.write(f'**Feature matrix:** {X.shape}  \n\n')
    f.write(f'## LightGBM CPCV\n\n')
    f.write(f'- Mean AUC across folds: **{lgbm_mean_auc:.3f}**\n')
    f.write(f'- Baseline (imbalance_ema logistic) AUC: **{baseline_auc:.3f}**\n')
    f.write(f'- Gate decision: **{"APPROVED" if report.approved else "REJECTED"}**\n\n')
    f.write('## Per-Tier Precision\n\n')
    if len(tier_table):
        f.write(tier_table.to_markdown(index=False))
        f.write('\n\n')
    f.write('## Feature Importances (top 10)\n\n')
    f.write(feat_imp.head(10).to_markdown(index=False))
    f.write('\n\n## Engine Latency\n\n')
    f.write(f'- Dry-run calls: {len(latencies)}\n')
    f.write(f'- p50: **{p50:.2f}ms** (target <50ms)\n')
    f.write(f'- p95: {p95:.2f}ms\n\n')
    f.write(f'## Model Export\n\n- Path: `{model_path}`\n- Reload round-trip: OK\n')
log.info('12 │ wrote %s', report_path)


12 │ niche gate: PASS — proceeding with export
12 │ wrote <path> (▒.▒▒ KB)
TemplateMatcher.set_covariance: input κ=▒.▒▒; diagonal-loaded with λ=▒.▒▒e+02 → new κ=▒.▒▒
reload_model: primary model loaded with 100 features
Loaded model v1_nq_fwd1m with 6 templates
12 │ engine.reload_model() OK; model_version = 'v1_nq_fwd1m'
12 │ promoted to CURRENT.json → v1_nq_fwd1m_20260607_0126/v1_nq_fwd1m.pkl
PatternLibrary saved: <path> (v16, 6 patterns)
12 │ realigned library to 6 promoted templates: ['depth_down', 'depth_up', 'imbalance_down', 'imbalance_up', 'lgbm_down', 'lgbm_up']
TemplateMatcher.set_covariance: input κ=▒.▒▒; diagonal-loaded with λ=▒.▒▒e+02 → new κ=▒.▒▒
reload_model: primary model loaded with 100 features
Loaded model v1_nq_fwd1m with 6 templates
12 │ engine.load_current_model() round-trip OK
12 │ wrote <path>
